# Stage 5 — LLaDA-MoE fine-tune on the GBSR-denoised LLMGPR social dataset (LLMGPR)

Next-POI recommendation, same recipe as the TSMC run of `stage6b_run2_server.ipynb`:
depth-regularised hyperbolic POI embeddings injected through the trained curvature-aware
adapter (`src/alignment.py`), LoRA on LLaDA-MoE, and the **Acc@t-guided multi-mask objective**
(`src/objective.py`): at every one of the `N_MASKS` diffusion slots the batch Acc@k is
measured and the oracle picks the best POI (the target if it is in the model's top-k, else
the model's own argmax) as the imitation target for that generation step.

**Dataset: `LLMGPR`** — 423,376 check-ins / 6,889 users / 14,402 POIs, real timestamps and
category taxonomy paths, plus the social layer -- here **denoised with GBSR** (KDD'24,
`src/denoise_social_gbsr.py`) before it entered group construction and the KG. Everything
this notebook needs is **committed in this repo** on the `llmgpr-pipeline` branch --
`data/llmgpr/` (check-in splits, metadata, the denoised `friendship_old_LLMGPR.csv`) and
`data/llmgpr/kg_denoised/` (embeddings, alignment triples, RotH checkpoint). No external
downloads besides the model.

---

## Setup

```bash
git clone -b llmgpr-pipeline https://github.com/yosrkharrat/Group-recommendation-for-next-poi.git
cd Group-recommendation-for-next-poi

pip install torch --index-url https://download.pytorch.org/whl/cu124   # match your CUDA
pip install -r requirements.txt

export HF_TOKEN=hf_...          # READ scope is enough; or put it in a .env at the repo root
jupyter lab notebooks/stage5_lbsn_finetune.ipynb
```

**No paths need editing.** `DATA_DIR` is auto-detected by content (it resolves to
`data/llmgpr/`), and `find()` locates the `data/llmgpr/kg_denoised/` artifacts. Outputs go to
`./outputs` (`STAGE6B_OUT_DIR` to change). Only `kg_denoised/` carries RotH/embedding output
on this branch -- if you ever also train a non-denoised baseline into a sibling `kg/`, give
its files different names, since `find()` matches by exact filename anywhere under `data/`
and does not know which of two identically-named files you meant.

## Verify these four lines, then leave it

```
RUN_PROFILE=full  SUBSAMPLE_FRAC=1.0  VAL_MAX=None  TEST_MAX=None
HF token OK (from environment) — user: ...
[hw] 80 GB VRAM -> QUANTIZE=False BATCH_SIZE=16 GRAD_ACCUM=2 GRAD_CKPT=False
[D1] emb file=poi_hyperbolic_embs_LLMGPR.npy  rho=+0.NNNN  monotonic=True
```

`[D1]` asserts the embeddings carry a radial hierarchy. **The gate is ρ > 0.30 AND
monotone; it aborts rather than train on flat embeddings.** Unlike the LBSN_NYC run, this
notebook does not hardcode an expected rho for LLMGPR -- read `train_roth.py`'s own printed
D1 verdict (also saved in `data/llmgpr/kg_denoised/roth_results.json`) for the value this
specific KG produced, and treat that logged number as the reference, not this cell.

## Set `RUN_PROFILE = "smoke"` for the first pass

~15 minutes, exercises every stage including the adapter and the friendship leakage guard.
Worth doing once before committing to the full run.

## What differs from the TSMC notebook — and what deliberately does not

| | TSMC (`stage6b`) | this notebook |
|---|---|---|
| `DATASET` | `NYC` | `LLMGPR` (env `STAGE6B_DATASET` still wins) |
| align files | hardcoded `_NYC` | follow `DATASET` (triple/relation count read from the vocab file at load time, no edit needed) |
| split | re-split 70/10/20 in-notebook | **`RESPLIT=False`** — the CSVs ship the per-user chronological 70/10/20 the KG and group examples were built against |
| `vocab.pkl` | loaded (never used) | dropped — it belongs to the TSMC token space |
| prompt | profile + history | identical, **plus an opt-in `[friends]` block** (`USE_SOCIAL_CONTEXT`) |
| group task | — | **§9b group prompt** over LLMGPR's *social* groups (`build_groups.py --group-source social`, self-built into `data/llmgpr/groups_social/` on first run) + **§11b group eval** with the same trained heads; `TRAIN_ON_GROUPS=False` keeps the individual run single-variable |

Prompt template, masked-diffusion SFT, Acc@t objective (`N_MASKS=10`, `ACC_TOP_K=3`,
`W_ORACLE=1.0`), restricted-logit eval, LoRA config and seed are all unchanged — except two
knobs that moved with the group work: `PROFILE_TOP_K` 5 → 10 (as on `lbsn-handoff`) and
`MAX_LEN` 1024 → **4096**. `lbsn-handoff` used 2048; LLMGPR needs double that, measured with
the real LLaDA tokenizer over all 121,290 group examples (p50 1,959 / p99 3,193 / max 3,792
tokens — 2048 would truncate 46.5% of them, and HF truncates from the *end*, eating the
`[group summary]` + `[current time]` + `[group next POI]` cue). Individual §9 prompts are
unaffected: `collate` pads to the batch max, not to `MAX_LEN`. Otherwise the run is
methodologically comparable to the TSMC anchor (Acc@1 0.1699 / MRR 0.2508 at 5% data) — but
there is **no LBSN anchor yet: this run creates it**. Run once with `USE_SOCIAL_CONTEXT=False`
(the default) to establish the anchor, then flip it for the social ablation.

## The social layer (`USE_SOCIAL_CONTEXT = True`)

Adds a `[friends]` block to the prompt: the pooled favourite POIs of the user's friends,
friends taken from `friendship_old_LLMGPR.csv` **only** (the before-period snapshot, GBSR-denoised) and
favourites counted over the **train split only**. `friendship_new_only` pairs are the
friendship-prediction eval set and partly an *effect* of co-visits — §2 loads them purely to
assert they are disjoint from what reaches prompts.

## Checkpointing

Unchanged: every `CKPT_EVERY_STEPS` steps and each epoch boundary to `outputs/ckpt_latest`
(plus `ckpt_best` / `ckpt_best_acc1`), `RESUME=True` auto-resumes, `PUSH_TO_HUB=True` mirrors
to a private HF repo.


## 0 · Environment

In [ ]:
import os
_req = next((q for q in ("requirements.txt", "../requirements.txt") if os.path.exists(q)), None)
if _req:
    print(f"installing from {_req}")
    get_ipython().system(f"pip install -q -r {_req}")
else:
    print("no requirements.txt found -- relying on the explicit pins in the next cell")

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 37.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 93.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 122.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 189.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 280.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 281.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 132.6 MB/s  0:00:040:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 149.5 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 155.8 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 150.8 MB/s  0:00:010:00:0100:01
   ━━━━━━━━

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch

_TORCH_PIN = torch.__version__

!pip install -q --no-cache-dir \
    "torch=={_TORCH_PIN}" \
    "transformers==4.46.3" \
    "tokenizers" \
    "peft==0.13.2" \
    "accelerate" \
    "bitsandbytes"

ERROR: Could not find a version that satisfies the requirement torch==2.13.0+cu130 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0)
ERROR: No matching distribution found for torch==2.13.0+cu130


In [1]:
import transformers
from transformers import AutoTokenizer, AutoModel
from transformers.modeling_utils import PreTrainedModel

assert transformers.__version__ == "4.46.3"


## 0b · Stage the pipeline modules

`alignment.py` and friends live in the repo's `src/`. Attaching a dataset does not make them
importable — `/kaggle/input` is not on `sys.path` and is read-only. This copies whatever the
attached dataset provides into a writable directory, then fills any gaps from the public repo so
the notebook cannot break just because a dataset snapshot is stale.


In [ ]:
import os, sys, glob, shutil, subprocess

CODE_STAGE = "/kaggle/working/code" if os.path.isdir("/kaggle/working") else "./_code"
REPO_RAW = ("https://raw.githubusercontent.com/yosrkharrat/"
            "Group-recommendation-for-next-poi/llmgpr-pipeline/src")   # this notebook's branch
NEEDED = ["alignment.py", "objective.py",   # objective.py: Acc@t-guided objective + HyperbolicScorer
          "build_groups.py", "affinity.py", "hyperbolic_group.py"]   # section 9b: group examples + consensus

os.makedirs(CODE_STAGE, exist_ok=True)

_STAGE_ABS = os.path.abspath(CODE_STAGE)

def _locate(fname):
    """Find a directory containing `fname`, never returning the staging dir itself.

    CODE_STAGE must be excluded: the notebook's cwd is /kaggle/working, so "./code" resolves to
    the very directory this cell writes into. Without the guard, a SECOND run of this cell finds
    the files it staged on the first run and shutil.copyfile raises SameFileError.
    """
    for root in ("/kaggle/input", "./src", "../src", ".", ".."):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if os.path.abspath(dirpath) == _STAGE_ABS:
                continue
            if fname in files:
                return dirpath
    return None

# 1. whatever the attached dataset already has (the pipeline notebook exports code_run/)
_src = _locate("alignment.py") or _locate("build_groups.py")
if _src:
    _n = 0
    for _f in glob.glob(os.path.join(_src, "*.py")):
        _t = os.path.join(CODE_STAGE, os.path.basename(_f))
        if os.path.abspath(_f) == os.path.abspath(_t):
            continue                     # belt-and-braces against self-copy
        # copyfile + chmod, NOT shutil.copy: copy() preserves /kaggle/input's read-only bits,
        # so re-running this cell would otherwise fail with PermissionError.
        shutil.copyfile(_f, _t)
        os.chmod(_t, 0o644)
        _n += 1
    print(f"from dataset : {_src}  ({_n} modules)")

# 2. fill gaps from the repo. A dataset snapshot goes stale whenever src/ changes; main does not.
for _need in NEEDED:
    if not os.path.exists(os.path.join(CODE_STAGE, _need)):
        try:
            subprocess.run(["wget", "-q", f"{REPO_RAW}/{_need}",
                            "-O", os.path.join(CODE_STAGE, _need)], check=True)
            print(f"from repo    : {_need}")
        except Exception as e:
            raise RuntimeError(
                f"{_need} is neither in an attached dataset nor fetchable ({e}). Either turn "
                f"Internet ON, or attach a dataset containing the repo's src/*.py."
            ) from e

if CODE_STAGE not in sys.path:
    sys.path.insert(0, CODE_STAGE)
print("staged       :", ", ".join(sorted(os.path.basename(x)
                                        for x in glob.glob(f"{CODE_STAGE}/*.py"))))
for _need in NEEDED:
    assert os.path.exists(os.path.join(CODE_STAGE, _need)), f"{_need} missing after staging"

In [2]:
# ── HF token + model pre-download (works on Kaggle AND on a plain server) ───────────────
# Kaggle Secrets are NOT environment variables. Ticking HF_TOKEN under Add-ons -> Secrets makes
# it readable ONLY via kaggle_secrets.UserSecretsClient(); os.environ.get("HF_TOKEN") stays
# empty. An earlier server port of this notebook dropped the UserSecretsClient branch, which is
# why an attached-and-enabled secret still failed here. Order: Kaggle Secrets -> environment ->
# .env, then fail loudly, because a silent miss turns into a confusing 403 much later.
import os, sys, time
import huggingface_hub
from huggingface_hub import snapshot_download

MAX_WORKERS = 4
MODEL_NAME = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"
MODEL_NAME_0A = MODEL_NAME

# Set in the first code cell, before any hub import. Verify rather than re-set: by now
# huggingface_hub is long imported and a late assignment cannot change its transport.
if os.environ.get("HF_HUB_DISABLE_XET") != "1":
    print("  WARNING: HF_HUB_DISABLE_XET was not set before the first huggingface_hub import.\n"
          "  The 14.7 GB download will likely stall near ~200 MB with no error. Restart the\n"
          "  kernel and run from the top -- setting it now cannot help.")
else:
    print("  xet transport disabled (avoids the ~200 MB Kaggle download stall)")

_token_src = None

# 1. Kaggle Secrets (Add-ons -> Secrets, box ticked). Must be read through the client.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
        if _tok:
            os.environ["HF_TOKEN"] = _tok.strip()
            _token_src = "Kaggle Secrets"
    except Exception as e:
        # Not on Kaggle, or the secret is not attached to THIS notebook. Both are fine here.
        print(f"  (no Kaggle secret: {type(e).__name__}: {e})")
elif os.environ.get("HF_TOKEN"):
    _token_src = "environment"

# 2. .env next to the notebook (plain-server convenience)
if not os.environ.get("HF_TOKEN"):
    try:
        from dotenv import load_dotenv
        load_dotenv()
        if os.environ.get("HF_TOKEN"):
            _token_src = ".env"
    except ImportError:
        pass

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set.\n"
        "  Kaggle : Add-ons -> Secrets -> add HF_TOKEN, and make sure the checkbox next to it "
        "is TICKED FOR THIS NOTEBOOK (a secret can exist account-wide but be unattached here).\n"
        "  Server : export HF_TOKEN=hf_... , or put HF_TOKEN=... in a .env file beside this "
        "notebook.\n"
        "READ access is sufficient unless PUSH_TO_HUB=True, which needs WRITE."
    )

# Verify the token actually works before starting a 15 GB download that would 403 at the end.
HF_TOKEN = os.environ["HF_TOKEN"]
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)   # older hub versions read this name
try:
    _who = huggingface_hub.whoami(token=HF_TOKEN)
    print(f"HF token OK (from {_token_src}) — user: {_who.get('name')}")
except Exception as e:
    raise RuntimeError(
        f"HF_TOKEN was found (from {_token_src}) but the Hub rejected it: {type(e).__name__}: {e}\n"
        "Regenerate it at https://huggingface.co/settings/tokens and update the secret."
    )

t0 = time.time()
MODEL_PATH = snapshot_download(
    MODEL_NAME,
    max_workers=MAX_WORKERS,
    token=HF_TOKEN,          # pass explicitly rather than relying on env pickup
)
print(f"Model downloaded to: {MODEL_PATH} ({(time.time()-t0)/60:.1f} min)")


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model downloaded to: /home/ucloud/.cache/huggingface/hub/models--inclusionAI--LLaDA-MoE-7B-A1B-Instruct/snapshots/783d3467f108d28ac0a78d3e41af16ab05cabd8d (0.0 min)


## 1 · Config

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
import os, glob

DATASET  = os.environ.get("STAGE6B_DATASET", "LLMGPR")

# DATA_DIR: honour the env var, else auto-detect the attached Kaggle dataset by CONTENT.
# "./data" is only right when running locally from the repo root.
def _autodetect_data_dir(dataset):
    need = {f"train_{dataset}.csv", f"poi_metadata_{dataset}.csv"}
    for root in ("/kaggle/input", "./data", "../data", ".", ".."):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if need <= set(files):
                return dirpath
    return "./data"

DATA_DIR = os.environ.get("STAGE6B_DATA_DIR") or _autodetect_data_dir(DATASET)
OUT_DIR  = os.environ.get("STAGE6B_OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"
assert MODEL_NAME == globals().get("MODEL_NAME_0A", MODEL_NAME)

EMB_CONDITION = "hyperbolic"
# Point at the DEPTH-REGULARISED embeddings from src/train_roth.py (D1 rho=+0.85), not the
# original flat ones (rho=+0.03). This must be the exact filename: find()'s fallback globs
# "poi_hyperbolic_embs*.npy" and sorts, so if BOTH files are attached the old one wins
# alphabetically and the run silently trains on embeddings with no radial hierarchy.
# Cell "2 · Load data" asserts the hierarchy is actually present, so a wrong file fails fast.
EMB_FILE      = f"poi_hyperbolic_embs_{DATASET}.npy"
CURVATURE_C   = 1.0

# Curvature-aware alignment
# with a trained ManifoldAwareAdapter + GeometryPreservingLoss (ranking + radius + KG-triple +
# spectral terms). See "## 6b" below. Flip to False to fall back to the old fixed projection.
USE_CURVATURE_ALIGNMENT = True
# These follow DATASET (the TSMC notebook hardcoded "_NYC" here — the one adaptation trap).
# LBSN_NYC: 132,394 triples / 5 relations; ALIGN_NUM_RELATIONS is read from the vocab file at
# load time, so the 6 -> 5 relation change needs no edit anywhere else.
ALIGN_TRIPLES_FILE  = f"poi_poi_triples_{DATASET}.pt"
ALIGN_RELVOCAB_FILE = f"poi_relation_vocab_{DATASET}.json"
ALIGN_HIDDEN_DIM, ALIGN_NUM_LAYERS, ALIGN_DROPOUT = 1024, 4, 0.1
ALIGN_EPOCHS, ALIGN_BATCH_SIZE, ALIGN_TRIPLE_SAMPLE = 500, 512, 256
ALIGN_LR, ALIGN_WEIGHT_DECAY = 1e-3, 1e-4
ALIGN_W_RANKING, ALIGN_W_RADIUS, ALIGN_W_TRIPLE, ALIGN_W_SPECTRAL = 1.0, 0.2, 1.0, 0.01
ALIGN_SPECTRAL_TOPM, ALIGN_SPECTRAL_START_EPOCH = 32, 100
assert not USE_CURVATURE_ALIGNMENT or EMB_CONDITION == "hyperbolic"

# Acc@t-guided multi-mask objective (src/objective.py). N_MASKS=1, W_ORACLE=0 is
# byte-identical to the old single-token CE. N_MASKS=10: the model ranks the true POI into
# a 10-slot list (the group task -- section 9b -- shares this setting).
USE_ACC_AT_T_OBJECTIVE = True
N_MASKS      = 10 if USE_ACC_AT_T_OBJECTIVE else 1
ACC_TOP_K    = 3
W_RANK       = 1.0                       # CE at slot 0 toward the true POI (matches eval)
W_ORACLE     = 1.0 if USE_ACC_AT_T_OBJECTIVE else 0.0
LABEL_SMOOTH = 0.0

# "both" = trainable Linear(H, N_POI) head  +  tied MLP path proj(h)·W_POI[j].
# The head terminstead keeps run-1's capacity (absolute numbers should be >= run 1); the tied term
# makes the FROZEN injected embeddings reach the output, so hyperbolic-vs-random stays a real
# ablation  of being fit from context alone.
# "all" also adds objective.py's HyperbolicScorer (ranks by geodesic distance, not dot product).

# Output scoring.
SCORING_MODE = "both"                    # "head" | "tied" | "both" | "all"
PROJ_HIDDEN  = None

# Dataset split. The LBSN_NYC CSVs already ship per-user chronological 70/10/20 (the same
# resplit_per_user as the rest of the pipeline), and the KG's train-only relations + the
# group examples were built against THAT assignment — re-splitting here would silently
# disagree with them (docs/LBSN_HANDOFF.md, rule 2). Keep the files' split.
RESPLIT   = False
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.10

# User profile in the prompt.
USE_PROFILE   = True
PROFILE_TOP_K = 10          # widened "most visited" / best-places list (was 5)
PROFILE_CATS  = 3
PROFILE_HOURS = 3

# Social context in the prompt — the LBSN dataset's differentiator. Friends come from
# friendship_old_{DATASET}.csv ONLY (the before-period snapshot); friendship_new_only is the
# friendship-prediction eval set and partly an EFFECT of the co-visits the model trains on,
# so a leakage guard in section 2 asserts it never reaches prompts. OFF by default so the
# first LBSN run is pipeline-identical to the TSMC anchor; flip for the social ablation.
USE_SOCIAL_CONTEXT = False
FRIENDS_MAX        = 5    # friends pooled per user
FRIEND_TOP_POIS    = 5    # pooled favourite POIs shown in the [friends] block

MASK_TOKEN_ID = 156895

HIST_LEN    = 15
MAX_LEN     = 4096          # was 1024; lbsn-handoff raised it to 2048 for section 9b, and
                            # LLMGPR needs 4096 -- MEASURED with the real LLaDA tokenizer over
                            # all 121,290 group examples: p50 1,959  p99 3,193  max 3,792 tokens.
                            # 2048 truncates 46.5% of them and 3072 still truncates 2.6%; 4096
                            # (cap 4,085 after N_MASKS) truncates NONE. LLMGPR prompts run long
                            # because its categories are full taxonomy paths and its 14,402 POIs
                            # mean 5-digit <poi_i> tokens -- the same knobs cost ~2x LBSN_NYC's
                            # tokens. Truncation here is not benign: HF truncates from the END,
                            # which would eat [group summary] + [current time] + the
                            # '[group next POI] ' cue, i.e. the instruction the model answers.
                            # Individual (section 9) batches are unaffected -- collate pads to the
                            # batch max, not to MAX_LEN. Group eval (section 11b) runs under
                            # @torch.no_grad(), so the longer sequences cost activations only; if
                            # you set TRAIN_ON_GROUPS=True, expect to halve BATCH_SIZE.

# ── Hardware profile: auto-detected, because the wrong one is an instant OOM ────────────
# LLaDA-MoE-7B is ~14.0 GB in bf16/fp16 and ~4.8 GB in 4-bit NF4. A Kaggle T4/P100 has
# ~15.8 GB, so the unquantized model leaves ~1.8 GB for activations + gradients + optimizer
# + the resized 168k-row embedding table (0.69 GB on its own) -> OOM every time. Quantizing
# is not optional below ~40 GB of VRAM. Set QUANTIZE explicitly to override the detection.
import torch as _torch
_vram_gb = (_torch.cuda.get_device_properties(0).total_memory / 1e9
            if _torch.cuda.is_available() else 0.0)
_big_gpu = _vram_gb >= 40.0            # B200 / A100-80 / H100 / L40S-48

QUANTIZE   = not _big_gpu              # 4-bit NF4 double-quant when VRAM is tight
BATCH_SIZE = 16 if _big_gpu else 4
GRAD_ACCUM = 2  if _big_gpu else 8     # effective batch 32 either way
GRAD_CKPT  = not _big_gpu              # trades ~25% speed for a large activation saving
print(f"[hw] {_vram_gb:.0f} GB VRAM -> QUANTIZE={QUANTIZE} BATCH_SIZE={BATCH_SIZE} "
      f"GRAD_ACCUM={GRAD_ACCUM} GRAD_CKPT={GRAD_CKPT}")

EPOCHS      = 3
LR          = 2e-5
HEAD_LR     = 3e-4
WARMUP_FRAC = 0.03

LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

SEED      = 42
LOG_EVERY = 25

# ── RUN PROFILE — the only knob that needs changing ─────────────────────────────────────
#   "full"  full training split, full val/test. THE REAL RUN. Needs a >=40 GB GPU; on a
#           16 GB Kaggle T4 this is ~30 h/epoch and cannot finish inside one session.
#   "probe" 5% subsample, val/test capped at 2,000. Reproduces the fast probe the existing
#           Acc@1=0.1699 anchor was trained on, so it is the like-for-like comparison.
#   "smoke" ~15 min end to end. Run once first to prove the whole path works.
RUN_PROFILE = "full"

if RUN_PROFILE == "full":
    SMOKE_TEST, SUBSAMPLE_FRAC, VAL_MAX, TEST_MAX = False, 1.0, None, None
elif RUN_PROFILE == "probe":
    SMOKE_TEST, SUBSAMPLE_FRAC, VAL_MAX, TEST_MAX = False, 0.05, 2000, 2000
elif RUN_PROFILE == "smoke":
    SMOKE_TEST, SUBSAMPLE_FRAC, VAL_MAX, TEST_MAX = True, 0.05, 200, 200
else:
    raise ValueError(f"RUN_PROFILE must be full|probe|smoke, got {RUN_PROFILE!r}")
print(f"RUN_PROFILE={RUN_PROFILE}  SUBSAMPLE_FRAC={SUBSAMPLE_FRAC}  "
      f"VAL_MAX={VAL_MAX}  TEST_MAX={TEST_MAX}")

# Fail loudly HERE rather than 5 hours into a run that cannot finish: "full" on a small GPU
# means BATCH_SIZE=4 and ~30 h/epoch, which exceeds a Kaggle session and most patience.
if RUN_PROFILE == "full" and 0 < _vram_gb < 40:
    raise RuntimeError(
        f"RUN_PROFILE='full' needs a >=40 GB GPU; this one has {_vram_gb:.0f} GB.\n"
        f"  At BATCH_SIZE={BATCH_SIZE} that is roughly 30 h/epoch (~90 h for 3 epochs).\n"
        f"  Use RUN_PROFILE='probe' (5% subsample, ~4.5 h, and the like-for-like comparison\n"
        f"  against the published Acc@1=0.1699 anchor), or 'smoke' for a 15-minute check.\n"
        f"  If you genuinely intend a multi-session full run, set RESUME=True and comment out\n"
        f"  this guard -- checkpoints in OUT_DIR/ckpt_latest will carry across restarts."
    )

CKPT_EVERY_STEPS = 200
RESUME           = True
HF_CKPT_REPO     = None
#  keep checkpoints under OUT_DIR/ckpt_<tag> only
PUSH_TO_HUB      = False

MANUAL_RESUME_DIR      = None
MANUAL_RESUME_EPOCH    = 1
MANUAL_RESUME_BEST_VAL = float("inf")

# ── Locating inputs ─────────────────────────────────────────────────────────────────────
# The inputs span MORE THAN ONE mounted dataset: the check-in CSVs come from the original data
# bundle (kushflq / poi-final), while the embeddings and alignment triples come from the stage
# 1-4 pipeline notebook's output. Searching only DATA_DIR finds the CSVs and misses the rest.
#
# os.walk rather than glob("**"): glob's recursive ** does not descend into symlinked
# directories, and Kaggle mounts datasets under /kaggle/input/datasets/<user>/<slug> in a way
# that can defeat it. os.walk with followlinks=True is what actually traverses these mounts.
# ".." last: it is only walked when every nearer root misses (e.g. kernel cwd=notebooks/).
SEARCH_ROOTS = [r for r in (DATA_DIR, "/kaggle/input", OUT_DIR, "./data", "../data", ".", "..")
                if r and os.path.isdir(r)]

def _walk_find(root, predicate, limit=None):
    out = []
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        for f in files:
            if predicate(f):
                out.append(os.path.join(dirpath, f))
                if limit and len(out) >= limit:
                    return out
    return out

def _inventory():
    """Every artifact-looking file under the mounted inputs -- printed when a lookup fails, so
    a missing/renamed/unattached dataset is obvious instead of guesswork."""
    exts = (".npy", ".pt", ".pkl", ".jsonl")
    seen, lines = set(), []
    for root in SEARCH_ROOTS:
        for pth in _walk_find(root, lambda f: f.endswith(exts) or f.endswith(".csv")):
            rp = os.path.realpath(pth)
            if rp in seen:
                continue
            seen.add(rp)
            try:
                mb = os.path.getsize(pth) / 1e6
            except OSError:
                mb = float("nan")
            lines.append(f"      {mb:8.2f} MB  {pth}")
    return lines[:60]

def find(fname):
    """Locate a file across every mounted input, DATA_DIR first."""
    if os.path.isabs(fname) and os.path.exists(fname):
        return fname

    base = os.path.basename(fname)
    for root in SEARCH_ROOTS:                       # exact filename wins, everywhere
        hits = sorted(_walk_find(root, lambda f, b=base: f == b, limit=1))
        if hits:
            return hits[0]

    # Prefix fallback -- deliberately noisy. This is how "poi_hyperbolic_embs.npy" used to
    # resolve to whichever poi_hyperbolic_embs*.npy sorted first, silently selecting the OLD
    # flat embeddings over the depth-regularised ones.
    stem, ext = os.path.splitext(base)
    for root in SEARCH_ROOTS:
        hits = sorted(_walk_find(root, lambda f, s=stem, e=ext: f.startswith(s) and f.endswith(e)))
        if hits:
            print(f"  WARNING: no exact match for {base!r}; prefix match found "
                  f"{[os.path.basename(h) for h in hits]} -> using {os.path.basename(hits[0])}. "
                  f"Verify this is the file you intended.")
            return hits[0]

    inv = _inventory()
    raise FileNotFoundError(
        f"{base!r} was not found under any of {SEARCH_ROOTS}.\n"
        f"  Attach the stage 1-4 pipeline notebook's output (Add Input -> Notebook Output), or a\n"
        f"  dataset containing kg/{base}. What IS mounted:\n" + "\n".join(inv or ["      (nothing)"])
    )

# A smoke test must shorten the alignment stage too: 500 epochs is ~4 min that prints only every
# 50, so a "15-minute" smoke run otherwise spends most of it apparently frozen.
if SMOKE_TEST:
    ALIGN_EPOCHS, ALIGN_SPECTRAL_START_EPOCH = 20, 10
    print("SMOKE_TEST: ALIGN_EPOCHS -> 20")

print(f"DATASET={DATASET}  EMB_CONDITION={EMB_CONDITION}  DATA_DIR={DATA_DIR}  OUT_DIR={OUT_DIR}")

## 1b · Server logging
Long unattended run on a server: mirror everything printed below to a timestamped log file under `OUT_DIR`, so the run's history survives even if the terminal / SSH session / Jupyter tab is closed. This did not exist in the Kaggle version because Kaggle keeps cell output itself.


In [ ]:
import sys, datetime

# Mirror stdout/stderr to a log file WITHOUT losing the notebook's own display.
#
# The trap: sys.__stdout__ is the kernel's *process* stdout. Under Jupyter, sys.stdout has been
# replaced by ipykernel's OutStream, and that is the object which renders inside the cell.
# Teeing sys.__stdout__ therefore sends every subsequent print to the Kaggle "Log" tab and the
# log file, while the cell itself stays completely blank -- a run that looks frozen but is fine.
# Tee the CURRENT sys.stdout instead.
class _Tee:
    def __init__(self, *streams):
        self.streams = streams
        self._is_tee = True
    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()
        return len(data)
    def flush(self):
        for s in self.streams:
            s.flush()
    def isatty(self):
        return False          # tqdm then emits plain lines instead of in-place updates

# Unwrap any previous Tee so re-running this cell nests nothing. Checked via an attribute rather
# than isinstance: re-running redefines the class, so isinstance against the new one would fail.
for _name in ("stdout", "stderr"):
    _cur = getattr(sys, _name)
    if getattr(_cur, "_is_tee", False):
        setattr(sys, _name, _cur.streams[0])

LOG_PATH = os.path.join(
    OUT_DIR, f"train_log_{DATASET}_{EMB_CONDITION}_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt")
_log_file = open(LOG_PATH, "a")
sys.stdout = _Tee(sys.stdout, _log_file)
sys.stderr = _Tee(sys.stderr, _log_file)
print(f"Logging this run to: {LOG_PATH}")
print("  (this line appearing in the CELL means the tee is wired correctly; if the cell is blank "
      "and only the Log tab fills up, the notebook is stale)")


Logging this run to: ./outputs/train_log_NYC_hyperbolic_20260806_140403.txt
POIs: 5120 | emb dim: 64 | condition: hyperbolic
[re-split] per-user chronological -> train=0.703  val=0.100  test=0.197 (target 0.70/0.10/0.20)
examples  train=102676  val=14719  test=29071
compat shim installed
MASK_TOKEN_ID=156895 (tokenizer agrees: True)
W_POI: (5120, 2048)  frozen=True  norm≈0.568  device=cuda:0
mixed-embedding wrapper installed (POI ids served from frozen W_POI)
LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 155,189,248 || all params: 7,532,914,688 || trainable%: 2.0601
SCORING_MODE='both'  tied MLP: 2048 -> 1024 -> 2048  trainable head params: 14.7M  (W_POI stays frozen: True)
You are a POI recommendation expert. Using the user's long-term profile and their recent check-ins, predict the next POI token.
[user profile]
check-ins so far: 1
most visited: <poi_3597> (Dining and Drinking > Restaurant > American Restaurant > New Ame

Please note that, unlike autoregressive models, LLaDA MoE employs a bidirectional attention mechanism. In the forward code in modeling_lladamoe.py, we set both attention_mask and causal_mask to None, which affects the default causal attention and causes the input attention_mask parameter to become ineffective. If you pass an attention mask and expect the model to use it for computing other attention mechanisms, it may lead to logits and aux_loss returned by the model being inconsistent with your expectations. 


e1 s800/6418 ema=4.0314 avg=0.0050 lora_lr=1.98e-05 head_lr=2.96e-04 12s gpu_free=113.88GiB
e1 s825/6418 ema=4.3241 avg=0.1485 lora_lr=1.97e-05 head_lr=2.96e-04 75s gpu_free=93.76GiB
e1 s850/6418 ema=4.4902 avg=0.2834 lora_lr=1.97e-05 head_lr=2.96e-04 125s gpu_free=83.67GiB
e1 s875/6418 ema=4.5917 avg=0.4106 lora_lr=1.97e-05 head_lr=2.95e-04 175s gpu_free=83.66GiB
e1 s900/6418 ema=4.7444 avg=0.5357 lora_lr=1.97e-05 head_lr=2.95e-04 229s gpu_free=83.66GiB
e1 s925/6418 ema=4.7891 avg=0.6511 lora_lr=1.96e-05 head_lr=2.94e-04 279s gpu_free=83.66GiB
e1 s950/6418 ema=4.7030 avg=0.7547 lora_lr=1.96e-05 head_lr=2.94e-04 335s gpu_free=73.29GiB
e1 s975/6418 ema=4.6196 avg=0.8498 lora_lr=1.96e-05 head_lr=2.94e-04 388s gpu_free=73.29GiB
e1 s1000/6418 ema=4.6372 avg=0.9445 lora_lr=1.95e-05 head_lr=2.93e-04 437s gpu_free=73.29GiB
e1 s1025/6418 ema=4.4324 avg=1.0217 lora_lr=1.95e-05 head_lr=2.93e-04 491s gpu_free=73.29GiB
e1 s1050/6418 ema=4.4759 avg=1.1059 lora_lr=1.95e-05 head_lr=2.92e-04 541s gpu_

/home/ucloud/.local/lib/python3.12/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


e1 s1200/6418 ema=4.3914 avg=1.5199 lora_lr=1.93e-05 head_lr=2.90e-04 879s gpu_free=73.25GiB
e1 s1225/6418 ema=4.4081 avg=1.5783 lora_lr=1.93e-05 head_lr=2.90e-04 935s gpu_free=73.25GiB
e1 s1250/6418 ema=4.3176 avg=1.6298 lora_lr=1.93e-05 head_lr=2.89e-04 987s gpu_free=73.25GiB
e1 s1275/6418 ema=4.2464 avg=1.6789 lora_lr=1.93e-05 head_lr=2.89e-04 1042s gpu_free=73.25GiB
e1 s1300/6418 ema=4.2056 avg=1.7270 lora_lr=1.92e-05 head_lr=2.88e-04 1100s gpu_free=73.25GiB
e1 s1325/6418 ema=4.2381 avg=1.7748 lora_lr=1.92e-05 head_lr=2.88e-04 1160s gpu_free=73.25GiB
e1 s1350/6418 ema=4.1823 avg=1.8176 lora_lr=1.92e-05 head_lr=2.88e-04 1216s gpu_free=73.25GiB
e1 s1375/6418 ema=4.1171 avg=1.8574 lora_lr=1.91e-05 head_lr=2.87e-04 1277s gpu_free=73.25GiB
e1 s1400/6418 ema=4.1495 avg=1.8984 lora_lr=1.91e-05 head_lr=2.87e-04 1332s gpu_free=73.25GiB
e1 s1425/6418 ema=4.0825 avg=1.9347 lora_lr=1.91e-05 head_lr=2.86e-04 1398s gpu_free=73.25GiB
e1 s1450/6418 ema=4.3258 avg=1.9822 lora_lr=1.91e-05 head_lr=2.

## 2 · Load data & artifacts

In [5]:
import numpy as np, pandas as pd, torch, random

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df = pd.read_csv(find(f"train_{DATASET}.csv"))
val_df   = pd.read_csv(find(f"val_{DATASET}.csv"))
test_df  = pd.read_csv(find(f"test_{DATASET}.csv"))
meta_df  = pd.read_csv(find(f"poi_metadata_{DATASET}.csv"))

# (The TSMC notebook loaded vocab.pkl here — a KG entity vocabulary nothing in this
# notebook reads, and whose token space does not exist for LBSN_NYC. Dropped.)

poi_embs = np.load(find(EMB_FILE))
N_POI = len(meta_df)
assert poi_embs.shape[0] == N_POI, f"emb rows {poi_embs.shape[0]} != n_poi {N_POI}"
print(f"POIs: {N_POI} | emb dim: {poi_embs.shape[1]} | condition: {EMB_CONDITION}")

for df in (train_df, val_df, test_df):
    if "utc_time" in df.columns:
        ts = pd.to_datetime(df["utc_time"], errors="coerce", utc=True)
        df["hour"] = ts.dt.hour.fillna(12).astype(int)
        df["dow"]  = ts.dt.day_name().fillna("Monday")
    else:
        df["hour"] = 12; df["dow"] = "Monday"

poi_cat = meta_df.set_index("poi_idx")["category"].fillna("Venue").to_dict()

# ── D1 guard: are these actually the depth-regularised embeddings? ──────────────────────
# The whole group-consensus argument needs taxonomy depth to predict hyperbolic radius. The
# shipped embeddings scored rho=+0.03 (ABSENT, flat radii); src/train_roth.py with the depth
# regulariser scores +0.85 (STRONG, monotonic). Loading the wrong .npy is silent and costs a
# full fine-tune, so it is checked here rather than discovered in the results.
#
# MEASURED on the committed LLMGPR file (data/llmgpr/kg_denoised/poi_hyperbolic_embs_LLMGPR.npy,
# recomputed with src/train_roth.py's own d1_radial_hierarchy -- kg_denoised/roth_results.json
# was not committed, so this is the reference number until it is):
#     rho = +0.3245  STRONG  monotone   mean radius d1=1.014 d2=1.017 d3=1.055 d4=1.073
# That PASSES, but by only 0.025 over the rho > 0.30 gate -- far weaker than LBSN_NYC (+0.63)
# or TSMC (+0.85), and the radii span just 1.014..1.073. Two reasons, both structural: 72.7%
# of LLMGPR POIs sit at taxonomy depth 2 (tie-capping rho), and this KG carries only 2
# POI-POI relations (IS_NEAR_TO, FOLLOWED_BY) against LBSN_NYC's 5, because the 10 km region
# catalogue covers ~43% of visited venues so proximity is built over that subset only.
# Treat +0.32 as the baseline to beat, not as a healthy number: if the hyperbolic-vs-random
# ablation comes out flat, a hierarchy this shallow is the first thing to suspect.
if EMB_CONDITION == "hyperbolic":
    _cat_col = "category_path" if "category_path" in meta_df.columns else "category"
    _depth = np.array([len([x for x in str(s).split(">") if x.strip()])
                       for s in meta_df[_cat_col].fillna("")], dtype=float)
    _r = np.linalg.norm(poi_embs, axis=1)
    def _rank(a):
        # Average tied ranks (matches src/train_roth.py's own d1_radial_hierarchy). A plain
        # argsort rank is WRONG when one variable has heavy ties -- taxonomy depth here has
        # only 4 distinct values with 73% of POIs at depth 2 on LLMGPR, and un-averaged ranks
        # inject spurious ordering noise inside that tie block that can swing rho by 0.25+.
        o = np.argsort(a, kind="mergesort"); rk = np.empty(len(a)); rk[o] = np.arange(len(a))
        _, inv, cnt = np.unique(a, return_inverse=True, return_counts=True)
        sums = np.zeros(len(cnt)); np.add.at(sums, inv, rk)
        return (sums / cnt)[inv]
    _rd, _rr = _rank(_depth), _rank(_r)
    _rho = float(np.corrcoef(_rd, _rr)[0, 1])
    _by = {int(d): float(_r[_depth == d].mean()) for d in sorted(set(_depth.tolist())) if (_depth == d).sum()}
    _mono = all(b > a for a, b in zip(list(_by.values()), list(_by.values())[1:]))
    print(f"[D1] emb file={os.path.basename(find(EMB_FILE))}  rho={_rho:+.4f}  monotonic={_mono}")
    print("     mean radius by taxonomy depth: " + "  ".join(f"d{k}={v:.3f}" for k, v in _by.items()))
    assert _rho > 0.30 and _mono, (
        f"these embeddings have no radial hierarchy (rho={_rho:+.4f}, monotonic={_mono}) -- "
        f"you have loaded the OLD poi_hyperbolic_embs.npy, not the depth-regularised "
        f"poi_hyperbolic_embs_{DATASET}.npy from src/train_roth.py")

# ── Social layer: before-period friendship snapshot + leakage guard ─────────────────────
FRIENDS = {}
if USE_SOCIAL_CONTEXT:
    fr_old = pd.read_csv(find(f"friendship_old_{DATASET}.csv"))       # the ONLY file prompts may use
    fr_new = pd.read_csv(find(f"friendship_new_only_{DATASET}.csv"))  # eval-only: loaded JUST to assert disjointness
    _pairs = lambda df: {tuple(sorted(p)) for p in df[["u1", "u2"]].itertuples(index=False)}
    _old, _new = _pairs(fr_old), _pairs(fr_new)
    assert not (_old & _new), "friendship_new_only pairs leaked into friendship_old!"
    del fr_new, _new
    for a, b in sorted(_old):
        FRIENDS.setdefault(int(a), []).append(int(b))
        FRIENDS.setdefault(int(b), []).append(int(a))
    print(f"friends: {len(_old)} before-period edges over {len(FRIENDS)} users "
          f"(new-only pairs asserted disjoint from prompt inputs)")


## 3 · Keep the shipped per-user splits + build (profile, history, target) examples


In [6]:
# ── Re-split 70/10/20 + build (profile, history, target) examples ──────────
from collections import Counter

for _df, _name in ((train_df, "train"), (val_df, "val"), (test_df, "test")):
    _df["split"] = _name
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
_orig_split = full_df["split"].copy()

# ── per-user chronological 70/10/20 ──────────────────────────────
def resplit_per_user(df, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Overwrite df['split'] with a per-user chronological 70/10/20 assignment."""
    df = df.sort_values(["user_id", "utc_time"] if "utc_time" in df.columns else ["user_id"],
                        kind="mergesort").reset_index(drop=True)
    splits = np.empty(len(df), dtype=object)
    for _, idx in df.groupby("user_id", sort=False).indices.items():
        n = len(idx)
        n_tr  = max(1, int(np.ceil(n * train_frac)))
        n_val = int(np.ceil(n * (train_frac + val_frac)))
        n_val = min(max(n_val, n_tr), n)
        splits[idx[:n_tr]]        = "train"
        splits[idx[n_tr:n_val]]   = "val"
        splits[idx[n_val:]]       = "test"
    df["split"] = splits
    return df

if RESPLIT:
    full_df = resplit_per_user(full_df)
    vc = full_df["split"].value_counts(normalize=True)
    print(f"[re-split] per-user chronological -> "
          f"train={vc.get('train',0):.3f}  val={vc.get('val',0):.3f}  test={vc.get('test',0):.3f} "
          f"(target {TRAIN_FRAC:.2f}/{VAL_FRAC:.2f}/{1-TRAIN_FRAC-VAL_FRAC:.2f})")

# ── CHANGE 1 · causal user profile ──────────────────────────────────────────
def _profile_from_prefix(poi_counter, cat_counter, hour_counter, n_seen):
    """Snapshot of everything the model is allowed to know about the user BEFORE the target."""
    return dict(
        n_seen   = n_seen,
        top_pois = poi_counter.most_common(PROFILE_TOP_K),
        top_cats = [c for c, _ in cat_counter.most_common(PROFILE_CATS)],
        top_hrs  = [h for h, _ in hour_counter.most_common(PROFILE_HOURS)],
    )

def build_split_examples(full_df, target_split, hist_len=HIST_LEN):
    ex = []
    for uid, g in full_df.groupby("user_id"):
        g = g.sort_values("utc_time") if "utc_time" in g.columns else g
        seq  = g["poi_idx"].tolist()
        hrs  = g["hour"].tolist()
        dows = g["dow"].tolist()
        splt = g["split"].tolist()

        poi_c, cat_c, hr_c = Counter(), Counter(), Counter()
        poi_c[seq[0]] += 1; cat_c[poi_cat.get(seq[0], "Venue")] += 1; hr_c[hrs[0]] += 1

        for i in range(1, len(seq)):
            if splt[i] == target_split:
                h = seq[max(0, i - hist_len):i]
                ex.append(dict(
                    user       = uid,
                    hist       = h,
                    hist_hours = hrs[max(0, i - hist_len):i],
                    profile    = _profile_from_prefix(poi_c, cat_c, hr_c, i),
                    target     = seq[i],
                    t_hour     = hrs[i],
                    t_dow      = dows[i],
                ))
            # only NOW does check-in i enter the counters, so it can never appear in its own profile
            poi_c[seq[i]] += 1
            cat_c[poi_cat.get(seq[i], "Venue")] += 1
            hr_c[hrs[i]] += 1
    return ex

train_ex = build_split_examples(full_df, "train")
val_ex   = build_split_examples(full_df, "val")
test_ex  = build_split_examples(full_df, "test")

# --- No example may list its own target as a past "most visited" POI
#     unless the user genuinely visited it earlier in the trajectory. We check the strictly
#     stronger, cheap invariant: profile counts must sum to <= n_seen (the prefix length).
for _ex in (train_ex[:1000] + val_ex[:1000] + test_ex[:1000]):
    assert sum(c for _, c in _ex["profile"]["top_pois"]) <= _ex["profile"]["n_seen"], "profile is not causal!"

if SMOKE_TEST:
    train_ex, val_ex, test_ex = train_ex[:200], val_ex[:50], test_ex[:50]
else:
    rng = random.Random(SEED)
    if SUBSAMPLE_FRAC < 1.0:
        train_ex = rng.sample(train_ex, max(1, int(len(train_ex) * SUBSAMPLE_FRAC)))
    # Seeded caps so val/test are the SAME subsample across runs -- an unseeded cap would make
    # two runs differ by sampling noise and look like a real effect.
    if VAL_MAX is not None and len(val_ex) > VAL_MAX:
        val_ex = random.Random(SEED).sample(val_ex, VAL_MAX)
    if TEST_MAX is not None and len(test_ex) > TEST_MAX:
        test_ex = random.Random(SEED + 1).sample(test_ex, TEST_MAX)
print(f"examples  train={len(train_ex)}  val={len(val_ex)}  test={len(test_ex)}")

# ── CHANGE 2 · pooled friend favourites for the [friends] prompt block ──────────────────
# Counted over the TRAIN split only. This is split-causal, not per-example timestamp-causal:
# a friend's train check-ins are that friend's chronologically first 70%, which precedes the
# eval period in aggregate but is not filtered against each target's exact timestamp. Val and
# test check-ins never enter it.
FRIEND_FAVES = {}
if USE_SOCIAL_CONTEXT:
    _tr = full_df[full_df["split"] == "train"]
    _user_cnt = {u: Counter(g["poi_idx"].tolist()) for u, g in _tr.groupby("user_id")}
    for _u, _frs in FRIENDS.items():
        _pool = Counter()
        for _fr in _frs[:FRIENDS_MAX]:
            _pool.update(_user_cnt.get(_fr, Counter()))
        if _pool:
            FRIEND_FAVES[_u] = dict(n_friends=len(_frs), top=_pool.most_common(FRIEND_TOP_POIS))
    print(f"friend context built for {len(FRIEND_FAVES)}/{len(FRIENDS)} users with friends")


## 4 · Hyperbolic → Euclidean projection utilities
`logmap₀` linearises the Poincaré ball at the origin (geometrically sound for a single global map),
a **fixed** linear projection lifts 64→`d_model`, and rows are **norm-matched** to the mean row norm
of the base embedding matrix so the injected vectors sit at the model's native scale. The same fixed
`W` is shared across all three conditions so the only thing that changes is the geometry of the input
`.npy` — the controlled comparison the paper hinges on.

In [7]:
import torch

def logmap0(x: torch.Tensor, c: float = 1.0, eps: float = 1e-9) -> torch.Tensor:
    """Poincare-ball log map at the origin. x: (N, d) with ||x|| < 1/sqrt(c)."""
    sqrt_c = c ** 0.5
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    max_norm = (1.0 - 1e-5) / sqrt_c
    x = torch.where(norm > max_norm, x / norm * max_norm, x)
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    return torch.atanh((sqrt_c * norm).clamp(max=1 - 1e-7)) * x / (sqrt_c * norm)

def build_injection(embs_np, d_model, condition, target_row_norm, seed=SEED):
    """Return (N_POI, d_model) fp32 rows to load into the frozen W_POI table."""
    g = torch.Generator().manual_seed(seed)
    if condition == "random":
        inj = torch.randn(embs_np.shape[0], d_model, generator=g)
    else:
        e = torch.tensor(embs_np, dtype=torch.float32)
        v = logmap0(e, c=CURVATURE_C) if condition == "hyperbolic" else e
        W = torch.randn(v.shape[1], d_model, generator=g) / (v.shape[1] ** 0.5)
        inj = v @ W
    inj = inj / inj.norm(dim=-1, keepdim=True).clamp_min(1e-9) * target_row_norm
    return inj  # (N_POI, d_model), float32

## 5 · transformers compatibility shim
LLaDA-family `trust_remote_code` classes predate a couple of attributes newer `transformers`
expects. This patch adds the missing `all_tied_weights_keys` so `from_pretrained` / `resize_token_embeddings`
don't trip. Harmless if the attribute already exists.

In [8]:
import transformers.modeling_utils as _mu
if not hasattr(_mu.PreTrainedModel, "all_tied_weights_keys"):
    _mu.PreTrainedModel.all_tied_weights_keys = {}
print("compat shim installed")

## 6 · Load model (4-bit) + extend vocabulary

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

MODEL_SRC = MODEL_PATH if "MODEL_PATH" in globals() else MODEL_NAME

# bf16 needs Ampere+ (sm_80). Turing T4 is sm_75 and has NO bf16 tensor cores, so fp16 there.
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available()
    and torch.cuda.get_device_capability(0)[0] >= 8
    and getattr(torch.cuda, "is_bf16_supported", lambda: True)()
    else torch.float16
)

# 4-bit NF4 + double quant: ~14.0 GB -> ~4.8 GB. Required on any GPU under ~40 GB; see the
# hardware block in the config cell. Skipping this is the single most common OOM cause here.
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
) if QUANTIZE else None
print(f"loading in {'4-bit NF4' if QUANTIZE else str(compute_dtype)}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SRC,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_SRC,
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    quantization_config=_quant_cfg,
    device_map={"": 0},
)

if QUANTIZE:
    # Casts layer norms / embeddings to fp32, disables the frozen params' grads and makes
    # gradient checkpointing work through the quantized layers. Without it, GRAD_CKPT=True
    # on a 4-bit model silently produces no gradients for the LoRA adapters.
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRAD_CKPT)

if torch.cuda.is_available():
    _free, _tot = torch.cuda.mem_get_info()
    print(f"after load: {(_tot-_free)/1e9:.2f} GB used / {_tot/1e9:.2f} GB total")

poi_tokens = [f"<poi_{i}>" for i in range(N_POI)]
tokenizer.add_tokens(poi_tokens, special_tokens=True)

model.resize_token_embeddings(
    len(tokenizer),
    mean_resizing=False,
)

POI_TOKEN_IDS = torch.tensor(tokenizer.convert_tokens_to_ids(poi_tokens))
POI_ID_START = int(POI_TOKEN_IDS.min())
POI_ID_END = int(POI_TOKEN_IDS.max())

assert POI_ID_END - POI_ID_START + 1 == N_POI
assert torch.equal(
    POI_TOKEN_IDS,
    torch.arange(POI_ID_START, POI_ID_END + 1),
)

_mask_from_tok = getattr(tokenizer, "mask_token_id", None)
if _mask_from_tok is not None and _mask_from_tok != MASK_TOKEN_ID:
    raise ValueError(
        f"MASK_TOKEN_ID={MASK_TOKEN_ID} in the config cell does not match "
        f"tokenizer.mask_token_id={_mask_from_tok}. Update MASK_TOKEN_ID."
    )
print(f"MASK_TOKEN_ID={MASK_TOKEN_ID} (tokenizer agrees: {_mask_from_tok == MASK_TOKEN_ID if _mask_from_tok is not None else 'no mask_token_id attr, unverified'})")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## 6b · Curvature-aware alignment training
Ported from the reference GeoPOI pipeline's `alignment/curvature_aware_alignment.py`
(`ManifoldAwareAdapter` + `GeometryPreservingLoss`), replacing `build_injection`'s fixed random
projection with a **trained** one. Runs entirely on the small `[N_POI, hyp_dim]` embedding table —
independent of the LLM's own forward pass — before `W_POI` is built in the next section.

Four loss terms, matching the reference recipe exactly (see `alignment.py`): neighborhood-ranking
(preserve hyperbolic nearest-neighbor order), radius/hierarchy preservation, KG-triple
preservation (needs real `(head, relation, tail)` POI-POI edges — crosswalked from the reference
project's own knowledge graph into our `poi_idx` space by `build_poi_poi_triples.py`, run once
ahead of time), and a spectral/eigenvalue term gated on only after `ALIGN_SPECTRAL_START_EPOCH`.


In [ ]:
# ── Curvature-aware alignment: setup ────────────────────────────────────────
import json
import os
import shutil
import sys
import time
import glob
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# src modules were staged onto sys.path in cell 0b.
from alignment import (
    ManifoldAwareAdapter, RadiusPredictor, RelationScorer,
    GeometryPreservingLoss, match_llm_embedding_distribution, geodesic_distance,
)

if USE_CURVATURE_ALIGNMENT:
    poi_poi_triples = torch.load(find(ALIGN_TRIPLES_FILE))
    with open(find(ALIGN_RELVOCAB_FILE)) as f:
        _relvocab = json.load(f)
    ALIGN_NUM_RELATIONS = len(_relvocab["relation_to_id"])

    assert poi_poi_triples.dim() == 2 and poi_poi_triples.shape[1] == 3
    assert poi_poi_triples[:, [0, 2]].min().item() >= 0
    assert poi_poi_triples[:, [0, 2]].max().item() < N_POI, "triple references an out-of-range poi_idx"
    print(f"POI-POI triples: {len(poi_poi_triples):,}  relations: {ALIGN_NUM_RELATIONS}")
else:
    print("USE_CURVATURE_ALIGNMENT=False — skipping alignment setup, build_injection will be used instead")


In [ ]:
# ── Curvature-aware alignment: training function ────────────────────────────
# Trains ManifoldAwareAdapter + RadiusPredictor + RelationScorer jointly
def run_geometry_alignment(hyp_embs_np, triples, num_relations, d_model, device,
                            epochs=ALIGN_EPOCHS, batch_size=ALIGN_BATCH_SIZE,
                            triple_sample=ALIGN_TRIPLE_SAMPLE):
    hyp_dim = hyp_embs_np.shape[1]
    n_pois = hyp_embs_np.shape[0]
    hyp_raw = torch.tensor(hyp_embs_np, dtype=torch.float32, device=device)
    tangent = logmap0(hyp_raw, c=CURVATURE_C)          # precomputed once, reused every epoch
    triples = triples.to(device)

    adapter = ManifoldAwareAdapter(hyp_dim, d_model, hidden_dim=ALIGN_HIDDEN_DIM,
                                    num_layers=ALIGN_NUM_LAYERS, dropout=ALIGN_DROPOUT).to(device)
    radius_predictor = RadiusPredictor(d_model).to(device)
    relation_scorer = RelationScorer(num_relations, d_model).to(device)
    geom_loss_fn = GeometryPreservingLoss(
        radius_predictor=radius_predictor, relation_scorer=relation_scorer,
        weight_ranking=ALIGN_W_RANKING, weight_radius=ALIGN_W_RADIUS,
        weight_triple=ALIGN_W_TRIPLE, weight_spectral=ALIGN_W_SPECTRAL,
        top_m_spectral=ALIGN_SPECTRAL_TOPM,
    ).to(device)

    trainable = (list(adapter.parameters()) + list(radius_predictor.parameters())
                 + list(relation_scorer.parameters()))
    optimizer = torch.optim.AdamW(trainable, lr=ALIGN_LR, weight_decay=ALIGN_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    loader = DataLoader(TensorDataset(hyp_raw, tangent, torch.arange(n_pois, device=device)),
                         batch_size=batch_size, shuffle=True)

    adapter.train(); radius_predictor.train(); relation_scorer.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        use_spectral = epoch > ALIGN_SPECTRAL_START_EPOCH
        for hyp_b, tan_b, _idx in loader:
            aligned_b = adapter(tan_b)
            _, loss_dict = geom_loss_fn(hyp_emb=hyp_b, aligned_emb=aligned_b,
                                         curvature=CURVATURE_C, use_spectral=use_spectral)

            # Triple loss: sampled independently of the DataLoader batch, every step.
            triple_idx = torch.randint(0, len(triples), (triple_sample,), device=device)
            ts = triples[triple_idx]
            h_aligned = adapter(logmap0(hyp_raw[ts[:, 0]], c=CURVATURE_C))
            t_aligned = adapter(logmap0(hyp_raw[ts[:, 2]], c=CURVATURE_C))
            neg_t_idx = torch.randint(0, n_pois, (triple_sample,), device=device)
            neg_t_aligned = adapter(logmap0(hyp_raw[neg_t_idx], c=CURVATURE_C))
            pos_score = relation_scorer.score(h_aligned, ts[:, 1], t_aligned)
            neg_score = relation_scorer.score(h_aligned, ts[:, 1], neg_t_aligned)
            triple_loss = F.relu(1.0 - pos_score + neg_score).mean()

            spectral_term = loss_dict.get("spectral", torch.tensor(0.0, device=device))
            total_loss = (ALIGN_W_RANKING  * loss_dict["ranking"] +
                          ALIGN_W_RADIUS   * loss_dict["radius"] +
                          ALIGN_W_SPECTRAL * spectral_term +
                          ALIGN_W_TRIPLE   * triple_loss)

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            epoch_loss += total_loss.item()

        scheduler.step()
        if epoch % 50 == 0 or epoch == epochs:
            spectral_str = f" spectral={loss_dict['spectral']:.4f}" if use_spectral else ""
            print(f"[align] epoch {epoch}/{epochs}  loss={epoch_loss/len(loader):.4f}  "
                  f"ranking={loss_dict['ranking']:.4f}  radius={loss_dict['radius']:.4f}  "
                  f"triple={triple_loss.item():.4f}{spectral_str}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

    adapter.eval(); radius_predictor.eval(); relation_scorer.eval()
    with torch.no_grad():
        raw_aligned = adapter(tangent)
    return raw_aligned, adapter, radius_predictor, relation_scorer


In [ ]:
# Curvature-aware alignment
if USE_CURVATURE_ALIGNMENT:
    _align_base_emb = model.get_input_embeddings()
    _align_d_model = _align_base_emb.weight.shape[1]
    _align_device = _align_base_emb.weight.device

    t0 = time.time()
    raw_aligned, _align_adapter, _align_radius_pred, _align_rel_scorer = run_geometry_alignment(
        poi_embs, poi_poi_triples, ALIGN_NUM_RELATIONS, _align_d_model, _align_device,
    )
    print(f"alignment training done in {time.time() - t0:.0f}s")

    inj_aligned = match_llm_embedding_distribution(
        raw_aligned.float(), _align_base_emb.weight[:POI_ID_START].float()
    )

    torch.save({
        "adapter": _align_adapter.state_dict(),
        "radius_predictor": _align_radius_pred.state_dict(),
        "relation_scorer": _align_rel_scorer.state_dict(),
        "relation_to_id": _relvocab["relation_to_id"],
    }, f"{OUT_DIR}/alignment_modules_{DATASET}.pt")

    # verification: neighbor-structure preservation (non-blocking prints/warnings)
    #
    # OOM WARNING -- do not "simplify" this back to
    #     geodesic_distance(hyp_t.unsqueeze(1), hyp_t.unsqueeze(0), c_t)
    # which is what it used to be. That broadcasts to [5120, 5120, 64] = 1.68e9 elements
    # = 6.7 GB per fp32 intermediate, and mobius_add allocates about six of them, so the
    # single line needs ~40 GB. It survives on a B200 and OOMs instantly on a 16 GB Kaggle GPU.
    #
    # The arccosh form of the Poincare distance,
    #     d_c(x,y) = (1/sqrt c) * arccosh(1 + 2c||x-y||^2 / ((1-c||x||^2)(1-c||y||^2)))
    # depends on the data only through ||x-y||^2, so it is a matmul: no [N, N, d] tensor at
    # all. Chunked over rows to keep even the [N, N] result bounded. Same numbers to ~1e-7
    # at these radii (verified in src/objective.py's self-check).
    def _geo_topk(Z, c_val, k=10, chunk=512):
        z2_all = (Z * Z).sum(-1)                                  # [N]
        idx_out = []
        for s in range(0, Z.shape[0], chunk):
            q = Z[s:s + chunk]
            q2 = (q * q).sum(-1, keepdim=True)                    # [b, 1]
            sq = (q2 + z2_all.unsqueeze(0) - 2.0 * (q @ Z.t())).clamp_min(0.0)
            den = (1.0 - c_val * q2).clamp_min(1e-15) * (1.0 - c_val * z2_all.unsqueeze(0)).clamp_min(1e-15)
            D = torch.arccosh((1.0 + 2.0 * c_val * sq / den).clamp_min(1.0 + 1e-7)) / c_val ** 0.5
            D[torch.arange(q.shape[0], device=Z.device), torch.arange(s, s + q.shape[0], device=Z.device)] = float("inf")
            idx_out.append(D.topk(k, largest=False).indices)
        return torch.cat(idx_out, 0)

    with torch.no_grad():
        hyp_t = torch.tensor(poi_embs, dtype=torch.float32, device=_align_device)
        hyp_top10 = _geo_topk(hyp_t, float(CURVATURE_C), k=10)

        D_a = torch.cdist(inj_aligned, inj_aligned, p=2)
        D_a.fill_diagonal_(float("inf"))
        aligned_top10 = D_a.topk(10, largest=False).indices
        del D_a

        overlap = torch.tensor([
            len(set(hyp_top10[i].tolist()) & set(aligned_top10[i].tolist())) / 10
            for i in range(N_POI)
        ]).mean().item()

    _align_cats = [poi_cat.get(i, "Venue") for i in range(N_POI)]
    def _purity_at10(nbr_idx):
        hits = sum(1 for i in range(N_POI) for j in nbr_idx[i].tolist() if _align_cats[j] == _align_cats[i])
        return hits / (N_POI * 10)

    _rng = torch.Generator().manual_seed(SEED)
    random_top10 = torch.randint(0, N_POI, (N_POI, 10), generator=_rng)
    print(f"[align-verify] overlap@10 (hyp vs aligned neighbors): {overlap:.4f}")
    print(f"[align-verify] purity@10  hyp={_purity_at10(hyp_top10.cpu()):.4f}  "
          f"aligned={_purity_at10(aligned_top10.cpu()):.4f}  random={_purity_at10(random_top10):.4f}")
    if overlap < 0.05:
        print("[align-verify] WARNING: overlap@10 is barely above the random-neighbor floor "
              "(~0.002) — the adapter may not have learned useful structure.")

    _tgt = _align_base_emb.weight[:POI_ID_START].float()
    print(f"[align-verify] inj_aligned mean={inj_aligned.mean():.4f} std={inj_aligned.std():.4f}  "
          f"(target base_emb mean={_tgt.mean():.4f} std={_tgt.std():.4f})")
else:
    print("USE_CURVATURE_ALIGNMENT=False — will use build_injection's fixed random projection instead")


## 7 · Option A · Frozen `W_POI` table + lookup-time substitution
This is the heart of the paper-faithful injection.

1. **Discover** the base input-embedding module (`model.get_input_embeddings()`), read `d_model` and the
   mean row-norm of the *original* (pre-POI) vocabulary for scale-matching.
2. **Build** a separate `nn.Embedding` `W_POI` of shape `(N_POI, d_model)` in fp16, load the projected
   RotH rows into it, and **freeze** it (`requires_grad_(False)`). This table is *not* quantized, so the
   hyperbolic geometry is preserved exactly.
3. **Wrap** the base embedding module's `forward` so that any id ≥ `POI_ID_START` is served from `W_POI`
   and every other id from the (4-bit) base table. Because this returns `inputs_embeds` fed to the
   transformer, no internal architecture is touched. LoRA later attaches to the attention/MLP linears,
   **not** to `W_POI` — so `W_POI` stays frozen throughout, exactly as the paper specifies.

In [ ]:
import torch.nn as nn

base_emb = model.get_input_embeddings()
d_model  = base_emb.weight.shape[1]
emb_device = base_emb.weight.device

with torch.no_grad():
    # mean row-norm over the ORIGINAL vocab (exclude the freshly-resized POI rows)
    orig_rows = base_emb.weight[:POI_ID_START].float()
    tgt_norm  = orig_rows.norm(dim=-1).mean().item()
    if USE_CURVATURE_ALIGNMENT:
        # trained curvature-aware adapter (see "## 6b" above) — distribution-matched to
        # base_emb's own mean/std per dimension, NOT rescaled to tgt_norm's per-row L2 norm like
        # the build_injection fallback below, so row norms won't all equal tgt_norm here.
        inj = inj_aligned.float()
    else:
        inj = build_injection(poi_embs, d_model, EMB_CONDITION, tgt_norm)   # (N_POI, d_model) fp32

# separate, full-precision, FROZEN POI table (matches the base model's compute dtype)
W_POI = nn.Embedding(N_POI, d_model, dtype=compute_dtype, device=emb_device)
with torch.no_grad():
    W_POI.weight.copy_(inj.to(compute_dtype))
W_POI.weight.requires_grad_(False)
print(f"W_POI: {tuple(W_POI.weight.shape)}  frozen={not W_POI.weight.requires_grad}  "
      f"norm≈{tgt_norm:.3f}  device={emb_device}")

# --- wrap the base embedding forward: POI ids -> W_POI, others -> base table ---
_POI_START = POI_ID_START
_POI_END   = POI_ID_END
_N_POI     = N_POI
_orig_emb_forward = base_emb.forward

def _mixed_embedding_forward(input_ids):
    # Route the contiguous POI id block to W_POI. Bounding on BOTH sides is essential:
    # the LLaDA-MoE mask id (156895) and other specials can sit near the top of the base vocab,
    # so a one-sided `>= START` check could misroute them. `[_POI_START, _POI_END]` is exact.
    is_poi   = (input_ids >= _POI_START) & (input_ids <= _POI_END)
    base_ids = torch.where(is_poi, torch.zeros_like(input_ids), input_ids)
    out = _orig_emb_forward(base_ids)                       # (B, L, d) from the base table
    if is_poi.any():
        poi_local = (input_ids[is_poi] - _POI_START).clamp_(0, _N_POI - 1)
        poi_vecs  = W_POI(poi_local).to(out.dtype)          # compute_dtype -> compute dtype
        out = out.clone()
        out[is_poi] = poi_vecs
    return out

base_emb.forward = _mixed_embedding_forward
print("mixed-embedding wrapper installed (POI ids served from frozen W_POI)")


## 8 · LoRA (auto-detected target modules)
Target-module names are read off the live model rather than hardcoded, so this works whether the MoE
uses Llama-style `q_proj/…/down_proj` or something bespoke. The frozen `W_POI` is external to the PEFT
model and receives no adapters — it stays frozen by construction.

In [11]:
# ── LoRA ────────────────────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model
import torch.nn as nn

linear_names = {name.split(".")[-1] for name, m in model.named_modules()
                if isinstance(m, nn.Linear)}
CANDIDATES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
              "att_proj", "attn_out", "ff_proj", "ff_out"]
targets = [t for t in CANDIDATES if t in linear_names]
assert targets, f"No known LoRA target names found in: {sorted(linear_names)}"
print("LoRA targets:", targets)

lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                      target_modules=targets, bias="none", task_type=None)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
_emb_after = model.get_input_embeddings()
assert _emb_after.forward is _mixed_embedding_forward or getattr(_emb_after, "forward", None) is _mixed_embedding_forward \
       or _emb_after is base_emb, "embedding wrapper lost after PEFT wrap"


In [ ]:
import torch.nn as nn
from objective import HyperbolicScorer

H = model.config.hidden_size
PROJ_H = PROJ_HIDDEN or (H // 2)

assert SCORING_MODE in ("head", "tied", "both", "all"), f"unknown SCORING_MODE {SCORING_MODE!r}"

# LLaDA's final hidden states are LARGE-magnitude. Feeding them raw into a fresh head makes the
# logits explode once the LR warms up -> loss spikes UP (8.5 -> 12+). A trainable LayerNorm rescales
# each hidden vector first, so the logit scale is controlled. At init (weight=1, bias=0) it is
# ~identity-in-scale, so with the zero-inits below the step-0 loss is still exactly ln(5120) ~ 8.54.
poi_norm = nn.LayerNorm(H).to(emb_device, dtype=torch.float32)

poi_head = nn.Linear(H, N_POI, bias=True).to(emb_device, dtype=torch.float32)
with torch.no_grad():
    poi_head.weight.zero_(); poi_head.bias.zero_()

poi_proj = nn.Sequential(
    nn.Linear(H, PROJ_H),
    nn.GELU(),
    nn.LayerNorm(PROJ_H),
    nn.Linear(PROJ_H, d_model, bias=False),
).to(emb_device, dtype=torch.float32)
with torch.no_grad():
    nn.init.xavier_uniform_(poi_proj[0].weight); poi_proj[0].bias.zero_()
    poi_proj[3].weight.zero_()

# "all": HyperbolicScorer ranks by geodesic distance on poi_embs; gated to 0 at init (keeps
# the ln(N_POI) step-0 loss).
poi_hyper = None
if SCORING_MODE == "all":
    poi_hyper = HyperbolicScorer(
        H, torch.tensor(poi_embs, dtype=torch.float32), c_init=CURVATURE_C,
    ).to(emb_device, dtype=torch.float32)

_W_POI_f = W_POI.weight.detach().float()

def poi_scores(h):
    """h: (num_targets, H) float -> (num_targets, N_POI) logits, per SCORING_MODE."""
    h = poi_norm(h)                                # scale control; the key stability fix
    if SCORING_MODE == "head":
        return poi_head(h)
    tied = poi_proj(h) @ _W_POI_f.t()              # depends on the frozen injected embeddings
    if SCORING_MODE == "tied":
        return tied
    out = poi_head(h) + tied
    if SCORING_MODE == "all":
        out = out + poi_hyper(h)
    return out

_head_modules = [poi_norm] + (
    [poi_head] if SCORING_MODE == "head" else
    [poi_proj] if SCORING_MODE == "tied" else
    [poi_head, poi_proj] if SCORING_MODE == "both" else
    [poi_head, poi_proj, poi_hyper]                # "all"
)
for _m in _head_modules:
    for _p in _m.parameters():
        _p.requires_grad_(True)

_n_head_p = sum(p.numel() for m in _head_modules for p in m.parameters())
print(f"SCORING_MODE={SCORING_MODE!r}  tied MLP: {H} -> {PROJ_H} -> {d_model}  "
      f"trainable head params: {_n_head_p/1e6:.1f}M  (W_POI stays frozen: {not W_POI.weight.requires_grad})")
if SCORING_MODE == "head":
    print("  WARNING: 'head' alone weakens the embedding ablation — run 2 is meant to use 'both'.")

## 9 · Prompt (`[user profile]` + opt-in `[friends]`) + masked-diffusion SFT collator
`[user profile]` (causal long-term preference) → `[recent check-ins]` (the same last-15
`<poi_i>` + category + hour block) → `[current time]` → instruction. The response is still the single
`<poi_target>`

Every profile field comes from the causal snapshot built in §3 (`seq[:i]`). Nothing in this cell may reach back
into the example's own target or its future

In [ ]:
# ── prompt building (WITH user profile) + collator (LEFT-PAD with EOS) ─────
from objective import make_multimask_collate

EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else (tokenizer.pad_token_id or 0)

def profile_text(pr):
    """Render the CAUSAL profile snapshot built in §3. Reads pr only — never the target."""
    if not pr["top_pois"]:
        return "[user profile]\nnew user, no prior check-ins\n"
    pois = ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, x{c})" for p, c in pr["top_pois"])
    lines = [f"[user profile]",
             f"check-ins so far: {pr['n_seen']}",
             f"most visited: {pois}"]
    if pr["top_cats"]:
        lines.append("favourite categories: " + ", ".join(map(str, pr["top_cats"])))
    if pr["top_hrs"]:
        lines.append("usual hours: " + ", ".join(f"{h}:00" for h in sorted(pr["top_hrs"])))
    return "\n".join(lines) + "\n"

def friends_text(ex):
    """Opt-in [friends] block. Reads FRIEND_FAVES only: friendship_old x train-split
    check-ins, built in section 3 — friendship_new_only can never appear here."""
    if not USE_SOCIAL_CONTEXT:
        return ""
    fv = FRIEND_FAVES.get(ex["user"])
    if not fv:
        return ""
    pois = ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, x{c})" for p, c in fv["top"])
    return (f"[friends]\n{fv['n_friends']} friends in the network\n"
            f"friends' favourite places: {pois}\n")

def prompt_text(ex):
    hist_lines = "\n".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, {h}:00)"
                           for p, h in zip(ex["hist"], ex["hist_hours"]))
    head = ("You are a POI recommendation expert. Using the user's long-term profile and their "
            "recent check-ins, predict the next POI token.\n")
    prof = profile_text(ex["profile"]) if USE_PROFILE else ""
    soc  = friends_text(ex)
    return (head + prof + soc +
            "[recent check-ins]\n" + hist_lines +
            f"\n[current time] {ex['t_dow']} {ex['t_hour']}:00\n[next POI] ")

def encode_example(ex):
    p_ids = tokenizer(prompt_text(ex), add_special_tokens=False,
                      truncation=True, max_length=MAX_LEN - N_MASKS - 1)["input_ids"]
    tgt_id = POI_ID_START + ex["target"]
    return p_ids, tgt_id

def _single_mask_collate(batch):
    enc = [encode_example(ex) for ex in batch]
    L = max(len(p) + 1 for p, _ in enc)
    input_ids = torch.full((len(batch), L), EOS_ID, dtype=torch.long)   # EOS filler
    labels    = torch.full((len(batch), L), -100,   dtype=torch.long)
    attn      = torch.ones((len(batch), L),          dtype=torch.long)
    for i, (p_ids, tgt) in enumerate(enc):
        n = len(p_ids)
        start = L - (n + 1)
        input_ids[i, start:start + n] = torch.tensor(p_ids)
        input_ids[i, start + n]       = MASK_TOKEN_ID
        labels[i, start + n]          = tgt
    return dict(input_ids=input_ids, labels=labels, attention_mask=attn)

# USE_ACC_AT_T_OBJECTIVE swaps in objective.py's multi-mask collator (N_MASKS slots, local
# 0..N_POI-1 target). N_MASKS=1 (off) matches the single-mask path's truncation budget exactly.
if USE_ACC_AT_T_OBJECTIVE:
    def _encode_local_target(ex):
        p_ids, tgt_id = encode_example(ex)
        return p_ids, tgt_id - POI_ID_START
    collate = make_multimask_collate(_encode_local_target, MASK_TOKEN_ID, EOS_ID,
                                      n_masks=N_MASKS, max_len=MAX_LEN)
else:
    collate = _single_mask_collate

_probe = train_ex[0]
print(prompt_text(_probe))
_lens = [len(encode_example(e)[0]) for e in train_ex[:200]]
print(f"prompt tokens over 200 examples: mean={np.mean(_lens):.0f}  max={max(_lens)}  (MAX_LEN={MAX_LEN})")
assert max(_lens) < MAX_LEN - N_MASKS - 1, "prompts are hitting the truncation cap — lower HIST_LEN or PROFILE_TOP_K"

## 9b · Group prompt (multi-member, LLMGPR)

Group version of section 9's prompt, driven by `src/build_groups.py --group-source social` output
(`data/llmgpr/groups_social/group_examples_{train,val,test}.jsonl`). `--group-source social` swaps
the *real-group* miner for **LLMGPR/CubeRec's own rule** — a connected component of *friends*
co-present at the same venue in the same 180-min bucket, over the GBSR-denoised
`friendship_old_LLMGPR.csv` **only** (never `friendship_new_only`, per §2's leakage guard) — so the
`occasional` compositions and the co-attendance ties come from *social* co-presence rather than
KCGRS anchor windows; the `established` (affinity-clique) and `random` regimes are built exactly as
before. Those jsonl files are gitignored (~300 MB) — if they aren't on disk, this cell builds them
itself (measured on LLMGPR: **~2 min wall, ~4.5 GB peak RAM**, 23.7M user pairs through the
affinity stage → 71,933 / 16,164 / 33,193 examples) so the notebook runs standalone on a fresh
server on the full dataset; if that build fails, it raises rather than silently substituting a
smaller stand-in. The loader reads `_GROUPS_DIR` directly rather than through `find()`, because
this repo also carries `data/groups/` (TSMC) and `data/lbsn/groups/` (LBSN_NYC) examples in other
POI id spaces.

Each member's block reuses `profile_text()` from section 9 verbatim (so it stays byte-identical,
`PROFILE_TOP_K=10`-wide), plus up to `GROUP_HIST_PER_MEMBER=30` of their own recent check-ins
(pulled from the group's joint history, built with a wide enough window — `--hist-len 90` — that
30 is actually reachable per member, not just a ceiling). `[group summary]` adds shared
categories/hours, a ranked "best places visited by the group" list (top `GROUP_TOP_POPULAR_K` by
summed visit count, so it's never empty just because nothing clears the share threshold), the
threshold-gated "several members visit" list, and the hyperbolic consensus radius from
`src/hyperbolic_group.py`'s `gyromidpoint` over each member's own top-POI RotH embeddings
(`poi_embs`, D1-validated in section 2).

**`MAX_LEN` must be 4096 on this dataset** (§1). Measured with the real LLaDA tokenizer over
all 121,290 examples: p50 1,959 / p99 3,193 / **max 3,792** tokens — `lbsn-handoff`'s 2048
truncates 46.5% and 3072 still truncates 2.6%, while 4096 truncates none. LLMGPR is longer
than LBSN_NYC at identical knobs because its categories are full taxonomy paths and its
14,402 POIs give 5-digit `<poi_i>` tokens. The cell's own truncation assert is the guard:
if it ever fires, lower `GROUP_HIST_PER_MEMBER` (measured: 20 → max 3,618, 10 → max 3,041)
rather than letting HF truncate from the end, which would eat `[group summary]`,
`[current time]` and the `[group next POI]` cue the model is supposed to answer.

The target is still one real `<poi_target>`, ranked into a 10-slot list via
`USE_ACC_AT_T_OBJECTIVE`/`N_MASKS=10` (section 1) — same objective as the individual task, just
fed group-shaped examples. This cell rebuilds `collate` to dispatch on `"members" in ex`, so
section 10's training loop and section 11's `evaluate()` accept `group_ex` transparently (see
section 11b). `TRAIN_ON_GROUPS` (default `False`) keeps the individual-only run a single-variable
comparison; flip it to fold group examples into `train_ex`.


In [ ]:
# ── group prompt building, wired to the REAL group task (LLMGPR) ──────────────
import json, subprocess, sys as _sys
from collections import Counter as _Counter
from hyperbolic_group import gyromidpoint, radius as hyp_radius, group_heterogeneity

GROUP_MAX_MEMBERS     = 6     # full per-member blocks are long; cap to stay under MAX_LEN
GROUP_HIST_PER_MEMBER = 30    # recent check-ins shown per member
GROUP_MIN_SHARE       = 2     # a signal is "shared" if >= this many members have it
GROUP_TOP_POPULAR_K   = 10    # ranked "best places visited by the group" list length
TRAIN_ON_GROUPS       = False # opt-in: mix group examples into train_ex (see bottom of this cell)
GROUP_TRAIN_FRAC      = 0.15  # if TRAIN_ON_GROUPS: how many, as a fraction of len(train_ex)
GROUP_HIST_LEN        = 90    # joint (all-members) history depth build_groups.py draws from --
                              # must be well above GROUP_HIST_PER_MEMBER or most members would
                              # never individually reach it (it's a shared, not a per-member, pool)

# --group-source social: the REAL groups (-> the "occasional" compositions + co-attendance ties)
# follow LLMGPR/CubeRec's rule -- friends co-present at one venue in one 180-min bucket -- over
# the GBSR-denoised friendship_old_{DATASET}.csv ONLY, never friendship_new_only (section 2's
# leakage guard); "established"/"random" regimes are unchanged. Own dir, matching
# build_kg_lbsn.py's documented --groups-dir for this track (STAGE6B_GROUPS_DIR overrides).
_GROUPS_DIR = os.environ.get("STAGE6B_GROUPS_DIR") or os.path.join(DATA_DIR, "groups_social")

# Which KCGRS regimes to build. `established` needs the dense [n,n] affinity matrices in
# affinity.py -- ~8 GB EACH at GOWALLA's 31,667 users (5 of them), so it OOMs on any normal
# machine and build_groups.py skips building them unless `established` is actually requested.
# Default keeps the LLMGPR/TSMC behaviour; GOWALLA must drop `established`.
_DEFAULT_REGIMES = {"GOWALLA": "occasional random"}.get(DATASET, "established occasional random")
_GROUP_REGIMES = os.environ.get("STAGE6B_GROUP_REGIMES", _DEFAULT_REGIMES).split()
print(f"[group-task] regimes for {DATASET}: {_GROUP_REGIMES}")
if not all(os.path.exists(os.path.join(_GROUPS_DIR, f"group_examples_{s}.jsonl"))
           for s in ("train", "val", "test")):
    print(f"[group-task] group_examples_*.jsonl not under {_GROUPS_DIR} -- building them now "
          f"(src/build_groups.py --group-source social, measured ~2 min wall / ~4.5 GB peak RAM on LLMGPR) so this notebook "
          f"runs standalone...")
    # CODE_STAGE (section 0b) holds the staged src/*.py set, so build_groups.py finds affinity.py
    # next to it; find() is only the fallback.
    _bg_script = os.path.join(CODE_STAGE, "build_groups.py")
    if not os.path.exists(_bg_script):
        _bg_script = find("build_groups.py")
    _bg = subprocess.run(
        [_sys.executable, _bg_script, "--data-dir", DATA_DIR, "--dataset", DATASET,
         "--out-dir", _GROUPS_DIR, "--no-resplit",
         "--group-source", "social", "--friendship", "old",
         "--friend-old", find(f"friendship_old_{DATASET}.csv"),
         "--regimes", *_GROUP_REGIMES,
         "--profile-top-k", str(PROFILE_TOP_K), "--hist-len", str(GROUP_HIST_LEN)],
        capture_output=True, text=True)
    print(_bg.stdout[-3000:])
    if _bg.returncode != 0:
        print(_bg.stderr[-3000:])
        raise RuntimeError(
            "[group-task] build_groups.py failed -- see its output above. There is NO LLMGPR "
            "sample to fall back on (data/groups/samples/ is the TSMC 5,120-POI token space), "
            "so this raises rather than silently training/evaluating on the wrong dataset.")
    print(f"[group-task] built group_examples_*.jsonl under {_GROUPS_DIR}")

# ── loading: src/build_groups.py's schema (data/groups/samples/README.md) ────────
# {"example_id", "anchor", "members": [uid,...], "hist"/"hist_hours"/"hist_owner" (JOINT,
#  time-ordered, one entry per check-in any member contributed), "member_profiles" (parallel to
#  "members", each the SAME causal-profile dict section 3 builds), "target", "t_hour", "t_dow",
#  ...}. This is NOT the individual-example shape (no per-member "profile"/"hist" keys, and
#  history is joint with an owner tag) -- the split-out below is required before
#  group_prompt_text can use it.

def _load_group_examples(split):
    # Read from _GROUPS_DIR directly, NOT via find(): find() matches by exact basename anywhere
    # under ./data, and this repo also carries data/groups/ (TSMC, 5,120-POI token space) and
    # data/lbsn/groups/ (LBSN_NYC) -- a silent hit on either would train on the wrong poi ids.
    path = os.path.join(_GROUPS_DIR, f"group_examples_{split}.jsonl")
    if not os.path.exists(path):
        raise RuntimeError(
            f"{path} not found. The self-bootstrap above should have built it -- check its "
            f"output for why it failed (likely a missing data/llmgpr CSV) and re-run this cell.")
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return recs


def _group_record_to_members(rec):
    """Split the group's JOINT hist/hist_hours (tagged by hist_owner) back into per-member
    (profile, hist, hist_hours) -- the shape group_prompt_text expects. Without this split, a
    member's rendered "recent check-ins" would silently show the whole group's history."""
    per_hist = {u: [] for u in rec["members"]}
    for p, h, u in zip(rec["hist"], rec["hist_hours"], rec["hist_owner"]):
        per_hist.setdefault(u, []).append((p, h))
    members = []
    for u, pr in zip(rec["members"], rec["member_profiles"]):
        mh = per_hist.get(u, [])
        members.append(dict(user=u, profile=pr,
                             hist=[p for p, _ in mh], hist_hours=[h for _, h in mh]))
    return members


# ── the hyperbolic consensus signal (src/hyperbolic_group.py) ────────────────────────
_poi_embs_t64 = torch.tensor(poi_embs, dtype=torch.float64)   # float64: near-boundary precision


def _member_signature(pr, c=CURVATURE_C):
    """One member's top-POI RotH embeddings collapsed to one point via weighted gyromidpoint."""
    pois = pr["top_pois"]
    if not pois:
        return None
    idx = torch.tensor([p for p, _ in pois], dtype=torch.long)
    w = torch.tensor([float(cnt) for _, cnt in pois], dtype=torch.float64)
    pts = _poi_embs_t64[idx].unsqueeze(0)                      # (1, k, d)
    return gyromidpoint(pts, w.unsqueeze(0), c=c).squeeze(0)   # (d,)


def _hyperbolic_consensus_line(members, c=CURVATURE_C):
    """Smaller consensus radius than every member = the group's shared taste is more general
    than any individual member's. Returns None for <2 usable signatures."""
    sigs = [s for s in (_member_signature(m["profile"], c) for m in members) if s is not None]
    if len(sigs) < 2:
        return None
    X = torch.stack(sigs).unsqueeze(0)                         # (1, k, d)
    g = gyromidpoint(X, c=c).squeeze(0)
    r_members = [float(hyp_radius(s, c)) for s in sigs]
    r_group = float(hyp_radius(g, c))
    dispersion = float(group_heterogeneity(X, c=c)["dispersion"].squeeze(0))
    if r_group < min(r_members):
        verdict = "more general than every member"
    elif r_group >= max(r_members):
        verdict = "as specific as its most specific member"
    else:
        verdict = "between members' specificity levels"
    return (f"hyperbolic consensus radius: {r_group:.3f} (members: "
            + ", ".join(f"{r:.3f}" for r in r_members)
            + f") -> consensus is {verdict}; dispersion={dispersion:.3f}")


def build_group_example(rec):
    """Computed once here (not per epoch): depends only on the fixed member profiles."""
    members = _group_record_to_members(rec)[:GROUP_MAX_MEMBERS]
    return dict(example_id=rec["example_id"], members=members, target=rec["target"],
                t_hour=rec["t_hour"], t_dow=rec["t_dow"], heterogeneity=rec.get("heterogeneity"),
                hyp_consensus_line=_hyperbolic_consensus_line(members))


group_ex = {split: [build_group_example(r) for r in _load_group_examples(split)]
            for split in ("train", "val", "test")}

for _split_recs in group_ex.values():
    for _gex in _split_recs[:1000]:
        for _m in _gex["members"]:
            _pr = _m["profile"]
            assert sum(c for _, c in _pr["top_pois"]) <= _pr["n_seen"], "group member profile is not causal!"

if SMOKE_TEST:
    group_ex = {k: v[:100] for k, v in group_ex.items()}
else:
    if VAL_MAX is not None and len(group_ex["val"]) > VAL_MAX:
        group_ex["val"] = random.Random(SEED + 10).sample(group_ex["val"], VAL_MAX)
    if TEST_MAX is not None and len(group_ex["test"]) > TEST_MAX:
        group_ex["test"] = random.Random(SEED + 11).sample(group_ex["test"], TEST_MAX)
print(f"[group-task] examples  train={len(group_ex['train'])}  val={len(group_ex['val'])}  "
      f"test={len(group_ex['test'])}")


def _member_profile_block(m, idx):
    """[member N] header, then section 9's profile_text (minus its own header line), then
    recent check-ins."""
    lines = [f"[member {idx}]"]
    if USE_PROFILE:
        body = profile_text(m["profile"]).splitlines()
        if body and body[0] == "[user profile]":
            body = body[1:]
        lines.extend(body)
    hist = list(zip(m["hist"], m["hist_hours"]))[-GROUP_HIST_PER_MEMBER:]
    if hist:
        lines.append("recent check-ins: "
                     + ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, {h}:00)" for p, h in hist))
    return "\n".join(lines)


def _group_summary(members, hyp_line=None):
    """Shared categories/hours, a ranked "best places" list (top GROUP_TOP_POPULAR_K by summed
    visit count -- always populated when any member has history), the threshold-gated "several
    members visit" list, and the hyperbolic consensus line."""
    cat_c, hr_c, poi_members, poi_total = _Counter(), _Counter(), {}, _Counter()
    for m in members:
        cat_c.update(set(m["profile"]["top_cats"]))       # count each member at most once
        hr_c.update(set(m["profile"]["top_hrs"]))
        for p, cnt in m["profile"]["top_pois"]:
            poi_members[p] = poi_members.get(p, 0) + 1     # how many members list p as top
            poi_total[p] += cnt                             # summed visit count across members
    shared_cats = [c for c, n in cat_c.most_common()     if n >= GROUP_MIN_SHARE]
    shared_hrs  = sorted(h for h, n in hr_c.items()       if n >= GROUP_MIN_SHARE)
    common_pois = sorted((p for p, n in poi_members.items() if n >= GROUP_MIN_SHARE),
                         key=lambda p: -poi_members[p])
    top_popular = [p for p, _ in poi_total.most_common(GROUP_TOP_POPULAR_K)]
    lines = ["[group summary]", f"size: {len(members)} members"]
    if shared_cats:
        lines.append("shared favourite categories: " + ", ".join(shared_cats))
    if shared_hrs:
        lines.append("shared usual hours: " + ", ".join(f"{h}:00" for h in shared_hrs))
    if top_popular:
        lines.append(f"best places visited by the group (top {len(top_popular)}, by total "
                     "visits): "
                     + ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, x{poi_total[p]})"
                                 for p in top_popular))
    if common_pois:
        lines.append("POIs several members already visit: "
                     + ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, by {poi_members[p]} members)"
                                 for p in common_pois))
    if not (shared_cats or shared_hrs or common_pois):
        lines.append("members have diverse tastes with little overlap")
    if hyp_line:
        lines.append(hyp_line)
    return "\n".join(lines)


def group_prompt_text(gex):
    """Header -> per-member [member N] blocks -> [group summary] -> [current time] ->
    instruction. Target stays a single <poi_target>, ranked into a 10-slot list."""
    members = gex["members"]
    head = ("You are a POI recommendation expert. The following group of users will visit one "
            "place together. Each member has a long-term profile and recent check-ins. Weigh "
            "what the members have in common -- shared categories, shared hours, the best places "
            "the group has visited, POIs several of them already visit, and the hyperbolic "
            "consensus below -- and rank the POIs the whole group is most likely to visit "
            "together next.\n")
    member_blocks = "\n\n".join(_member_profile_block(m, i + 1) for i, m in enumerate(members))
    summary = _group_summary(members, hyp_line=gex.get("hyp_consensus_line"))
    return (head + "\n" + member_blocks + "\n\n" + summary
            + f"\n[current time] {gex['t_dow']} {gex['t_hour']}:00\n[group next POI] ")


def encode_group_example(gex):
    p_ids = tokenizer(group_prompt_text(gex), add_special_tokens=False,
                      truncation=True, max_length=MAX_LEN - N_MASKS - 1)["input_ids"]
    tgt_id = POI_ID_START + gex["target"]
    return p_ids, tgt_id


def is_group_example(ex):
    return "members" in ex


def encode_any_example(ex):
    return encode_group_example(ex) if is_group_example(ex) else encode_example(ex)


def _encode_local_target_any(ex):
    p_ids, tgt_id = encode_any_example(ex)
    return p_ids, tgt_id - POI_ID_START


def _single_mask_collate_any(batch):
    enc = [encode_any_example(ex) for ex in batch]
    L = max(len(p) + 1 for p, _ in enc)
    input_ids = torch.full((len(batch), L), EOS_ID, dtype=torch.long)
    labels    = torch.full((len(batch), L), -100,   dtype=torch.long)
    attn      = torch.ones((len(batch), L),          dtype=torch.long)
    for i, (p_ids, tgt) in enumerate(enc):
        n = len(p_ids)
        start = L - (n + 1)
        input_ids[i, start:start + n] = torch.tensor(p_ids)
        input_ids[i, start + n]       = MASK_TOKEN_ID
        labels[i, start + n]          = tgt
    return dict(input_ids=input_ids, labels=labels, attention_mask=attn)


# section 10's training loop and section 11's evaluate() both read `collate` by NAME at call
# time, so overwriting it here makes group_ex usable wherever train_ex/val_ex/test_ex already
# are -- see section 11b.
if USE_ACC_AT_T_OBJECTIVE:
    collate = make_multimask_collate(_encode_local_target_any, MASK_TOKEN_ID, EOS_ID,
                                      n_masks=N_MASKS, max_len=MAX_LEN)
else:
    collate = _single_mask_collate_any
print("[group-task] collate rebuilt to dispatch individual vs. group examples")

if TRAIN_ON_GROUPS and group_ex["train"]:
    _n_add = min(len(group_ex["train"]), max(1, int(len(train_ex) * GROUP_TRAIN_FRAC)))
    _add = random.Random(SEED + 12).sample(group_ex["train"], _n_add)
    train_ex = train_ex + _add
    print(f"[group-task] TRAIN_ON_GROUPS=True -- mixed {len(_add)} group examples into train_ex "
          f"-> {len(train_ex)} total.")
else:
    print("[group-task] TRAIN_ON_GROUPS=False -- group_ex is built for section 11b evaluation "
          "only; train_ex is unchanged.")

# ── real demo + truncation-budget check (mirrors section 9's own check) ────────────────
if group_ex["train"]:
    _gprobe = group_ex["train"][0]
    print("\n" + group_prompt_text(_gprobe))
    _probe_n = min(len(group_ex["train"]), 2000)
    _glens = [len(encode_group_example(g)[0]) for g in group_ex["train"][:_probe_n]]
    print(f"\n[group-prompt] tokens over {_probe_n} examples: mean={np.mean(_glens):.0f} "
          f"max={max(_glens)}  (MAX_LEN={MAX_LEN})")
    assert max(_glens) < MAX_LEN - N_MASKS - 1, (
        "group prompts are hitting the truncation cap -- lower GROUP_MAX_MEMBERS, "
        "GROUP_HIST_PER_MEMBER or PROFILE_TOP_K, or raise MAX_LEN")
else:
    print("[group-task] no group examples loaded for 'train' -- group path is defined but inert.")


## 10 · Training loop (custom masked-diffusion SFT)

In [ ]:
# ── training loop ───────────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from huggingface_hub import HfApi, create_repo, snapshot_download, whoami
from peft.utils.save_and_load import set_peft_model_state_dict
from safetensors.torch import load_file as load_safetensors
from objective import multimask_loss, gather_slot_hidden
import torch.nn.functional as F, math, time

# Optional gradient checkpointing (trade ~20-30% speed for a big activation-memory cut).
if GRAD_CKPT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()          # needed so grads flow through the frozen embeddings
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    print("gradient checkpointing: ON")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free before training: {free/1e9:.2f} / {total/1e9:.2f} GiB")

train_loader = DataLoader(train_ex, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)

lora_params = [p for p in model.parameters() if p.requires_grad]
head_params = [p for m in _head_modules for p in m.parameters()]
trainable   = lora_params + head_params
opt = torch.optim.AdamW([
    {"params": lora_params, "lr": LR},
    {"params": head_params, "lr": HEAD_LR},
])

n_epochs = 1 if SMOKE_TEST else EPOCHS
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * n_epochs
_warmup_steps = int(WARMUP_FRAC * total_steps)
sched = get_linear_schedule_with_warmup(opt, _warmup_steps, total_steps)
best_val = float("inf")
best_val_acc1 = float("-inf")   # separate accuracy-selected best; loss and Acc@1 have diverged
                                 # before (docs §6.2). Not persisted across resumes.

# ── Pre-flight: make the run size, the warmup and the ETA explicit ──────────────────────
# Two things here have burned real hours. (1) WARMUP_FRAC is a fraction of TOTAL steps, so 20x
# more data means a 20x longer warmup: on the full split the LR stays under 1% of target for
# ~2,300 batch steps, the loss sits at ln(N_POI)=8.54, and it looks like a broken model rather
# than a schedule. (2) SUBSAMPLE_FRAC not taking effect is silent -- the only symptom is a step
# count 20x larger than expected, which is easy to miss.
_spe = len(train_loader)
_warm_batches = _warmup_steps * GRAD_ACCUM
print(f"\n  train examples   {len(train_ex):,}   (SUBSAMPLE_FRAC={SUBSAMPLE_FRAC})")
print(f"  batch steps/epoch {_spe:,}  x {n_epochs} epochs   optimizer steps {total_steps:,}")
print(f"  warmup            {_warmup_steps:,} optimizer steps = {_warm_batches:,} batch steps")
print(f"                    -> the loss stays near ln({N_POI})={math.log(N_POI):.4f} until then; "
      f"that is the schedule, not a stalled model")
if SUBSAMPLE_FRAC >= 1.0 and not SMOKE_TEST:
    print(f"  WARNING: SUBSAMPLE_FRAC=1.0 (full data). At ~7 s/step this is ~{_spe*7/3600:.0f} h "
          f"PER EPOCH, or ~{_spe*7*n_epochs/3600:.0f} h total -- far past Kaggle's 12 h session\n"
          f"           limit. Set SUBSAMPLE_FRAC=0.05 in the config cell and RE-RUN section 3 so\n"
          f"           the example lists are rebuilt, then restart training.")
print()

def _dev():
    return next(p.device for p in model.parameters() if p.requires_grad)

def _backbone():
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    return base.model

def _hidden_states(input_ids):
    out = _backbone()(input_ids=input_ids)
    return out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]   # (B, L, H)

def _forward_multimask(batch):
    """-> (logits [B, M, N_POI], target [B] LOCAL 0..N_POI-1)."""
    dev = _dev()
    ids      = batch["input_ids"].to(dev)
    mask_pos = batch["mask_pos"].to(dev)
    target   = batch["target"].to(dev)
    h = _hidden_states(ids)                                    # (B, L, H)
    B, M = mask_pos.shape
    h_slots = gather_slot_hidden(h, mask_pos).float()          # (B, M, H)
    logits  = poi_scores(h_slots.reshape(B * M, -1)).reshape(B, M, -1)
    return logits, target

def _forward_single(batch):
    """-> (logits [num_targets, N_POI], target [num_targets] LOCAL 0..N_POI-1). No slot dim."""
    dev = _dev()
    ids    = batch["input_ids"].to(dev)
    labels = batch["labels"].to(dev)
    h = _hidden_states(ids)                                    # (B, L, H)
    mask = labels != -100                                      # target positions (exactly one per row)
    h_tgt = h[mask].float()                                    # (num_targets, H)
    logits = poi_scores(h_tgt)                                 # (num_targets, N_POI)  <-- SCORING_MODE, matches eval
    target = labels[mask] - POI_ID_START                       # 0..N_POI-1
    return logits, target

def batch_loss(batch):
    """-> (loss, stats). stats is {} unless USE_ACC_AT_T_OBJECTIVE."""
    if USE_ACC_AT_T_OBJECTIVE:
        logits, target = _forward_multimask(batch)
        return multimask_loss(logits, target, w_rank=W_RANK, w_oracle=W_ORACLE,
                              top_k=ACC_TOP_K, label_smoothing=LABEL_SMOOTH)
    logits, target = _forward_single(batch)
    return F.cross_entropy(logits, target), {}

@torch.no_grad()
def _val_pass(examples):
    """One pass per batch -> loss + restricted-logit ranking metrics, read from slot 0."""
    model.eval()
    loader = DataLoader(examples, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    total_loss, n_batches, ranks = 0.0, 0, []
    for batch in loader:
        if USE_ACC_AT_T_OBJECTIVE:
            logits, target = _forward_multimask(batch)
            loss, _ = multimask_loss(logits, target, w_rank=W_RANK, w_oracle=W_ORACLE,
                                     top_k=ACC_TOP_K, label_smoothing=LABEL_SMOOTH)
            rank_logits, tgt = logits[:, 0, :], target         # slot 0 = the ranked position
        else:
            rank_logits, tgt = _forward_single(batch)
            loss = F.cross_entropy(rank_logits, tgt)
        r = (rank_logits > rank_logits.gather(1, tgt[:, None])).sum(1) + 1
        ranks.extend(r.tolist())
        total_loss += loss.item(); n_batches += 1
    ranks_t = torch.tensor(ranks, dtype=torch.float)
    return dict(n=len(ranks), loss=total_loss / max(n_batches, 1),
                acc1=(ranks_t <= 1).float().mean().item(),
                acc5=(ranks_t <= 5).float().mean().item(),
                acc10=(ranks_t <= 10).float().mean().item(),
                mrr=(1.0 / ranks_t).mean().item())

def evaluate(examples, name="test"):
    res = _val_pass(examples)
    print(f"[{name}] n={res['n']}  Acc@1={res['acc1']:.4f}  Acc@5={res['acc5']:.4f}  "
          f"Acc@10={res['acc10']:.4f}  MRR={res['mrr']:.4f}  loss={res['loss']:.4f}")
    return res

_hf_token = os.environ.get("HF_TOKEN")
if PUSH_TO_HUB:
    _hf_api = HfApi(token=_hf_token)
    if HF_CKPT_REPO is None:
        HF_CKPT_REPO = (f"{whoami(token=_hf_token)['name']}/"
                        f"llada-moe-run2-{DATASET}-{EMB_CONDITION}-ckpt")
    create_repo(HF_CKPT_REPO, token=_hf_token, private=True, exist_ok=True)
    print(f"checkpoint repo: hf.co/{HF_CKPT_REPO}  (tags: latest, best, best_acc1)")
else:
    print(f"PUSH_TO_HUB=False — checkpoints stay local under {OUT_DIR}/ckpt_<tag>")

def save_checkpoint(save_dir, epoch, step, best_val_):
    """Save adapter + head + optimizer/scheduler state — everything needed to resume exactly."""
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)                              # LoRA adapter
    head_state = {"scoring_mode": SCORING_MODE,
                  "poi_norm": poi_norm.state_dict(),
                  "poi_head": poi_head.state_dict(),
                  "poi_proj": poi_proj.state_dict()}
    if poi_hyper is not None:
        head_state["poi_hyper"] = poi_hyper.state_dict()          # SCORING_MODE="all" only
    torch.save(head_state, f"{save_dir}/poi_head.pt")             # trained output head(s)
    torch.save({"optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "epoch": epoch, "step": step, "best_val": best_val_},
               f"{save_dir}/trainer_state.pt")

def push_checkpoint(save_dir, tag):
    """Upload save_dir to the HF Hub under tag ('latest', 'best' or 'best_acc1') -- this is the durable copy."""
    _hf_api.upload_folder(folder_path=save_dir, repo_id=HF_CKPT_REPO, path_in_repo=tag,
                           ignore_patterns=["README.md"])
    print(f"  ↳ pushed '{tag}' checkpoint to hf.co/{HF_CKPT_REPO}")

def checkpoint(tag, epoch, step, best_val_):
    d = f"{OUT_DIR}/ckpt_{tag}"
    save_checkpoint(d, epoch, step, best_val_)
    if PUSH_TO_HUB:
        push_checkpoint(d, tag)

def _load_checkpoint_state(ckpt_dir):
    """Load adapter + head + optimizer/scheduler state from ckpt_dir; returns (epoch, step, best_val)."""
    set_peft_model_state_dict(model, load_safetensors(f"{ckpt_dir}/adapter_model.safetensors"))
    head_sd = torch.load(f"{ckpt_dir}/poi_head.pt", map_location=_dev())
    poi_norm.load_state_dict(head_sd["poi_norm"]); poi_head.load_state_dict(head_sd["poi_head"])
    poi_proj.load_state_dict(head_sd["poi_proj"])
    if poi_hyper is not None and "poi_hyper" in head_sd:
        poi_hyper.load_state_dict(head_sd["poi_hyper"])
    state = torch.load(f"{ckpt_dir}/trainer_state.pt", map_location=_dev())
    opt.load_state_dict(state["optimizer"]); sched.load_state_dict(state["scheduler"])
    return state["epoch"], state["step"], state["best_val"]

resume_epoch, resume_step = 1, 0
_resumed = False
if RESUME:
    local_ckpt_dir = f"{OUT_DIR}/ckpt_latest"
    if os.path.exists(f"{local_ckpt_dir}/trainer_state.pt"):
        resume_epoch, resume_step, best_val = _load_checkpoint_state(local_ckpt_dir)
        print(f"resumed from local checkpoint {local_ckpt_dir}: epoch {resume_epoch} step {resume_step} "
              f"best_val={best_val:.4f}")
        _resumed = True
    elif PUSH_TO_HUB:
        try:
            snap = snapshot_download(HF_CKPT_REPO, allow_patterns=["latest/*"], token=_hf_token)
            ckpt_dir = f"{snap}/latest"
            assert os.path.exists(f"{ckpt_dir}/trainer_state.pt")
            resume_epoch, resume_step, best_val = _load_checkpoint_state(ckpt_dir)
            print(f"resumed from hf.co/{HF_CKPT_REPO}: epoch {resume_epoch} step {resume_step} "
                  f"best_val={best_val:.4f}")
            _resumed = True
        except Exception as e:
            print(f"no HF checkpoint to resume from ({type(e).__name__}: {e})")

if not _resumed and MANUAL_RESUME_DIR and os.path.exists(f"{MANUAL_RESUME_DIR}/adapter_model.safetensors"):
    set_peft_model_state_dict(model, load_safetensors(f"{MANUAL_RESUME_DIR}/adapter_model.safetensors"))
    head_sd = torch.load(f"{MANUAL_RESUME_DIR}/poi_head.pt", map_location=_dev())
    poi_norm.load_state_dict(head_sd["poi_norm"]); poi_head.load_state_dict(head_sd["poi_head"])
    poi_proj.load_state_dict(head_sd["poi_proj"])
    if poi_hyper is not None and "poi_hyper" in head_sd:
        poi_hyper.load_state_dict(head_sd["poi_hyper"])
    resume_epoch, resume_step, best_val = MANUAL_RESUME_EPOCH, 0, MANUAL_RESUME_BEST_VAL
    print(f"bootstrapped from local checkpoint {MANUAL_RESUME_DIR} -> "
          f"resuming at epoch {resume_epoch}, best_val={best_val:.4f}")
    checkpoint("latest", resume_epoch, resume_step, best_val)   # persist immediately (local, +Hub if PUSH_TO_HUB)
    _resumed = True

if not _resumed:
    print("no checkpoint found anywhere — starting fresh from epoch 1")

if resume_epoch > n_epochs:
    print(f"checkpoint is already past n_epochs={n_epochs} — nothing left to train")

global_step = 0
for epoch in range(resume_epoch, n_epochs + 1):
    model.train(); t0 = time.time(); running = 0.0; ema = None
    skip_until = resume_step if epoch == resume_epoch else 0
    for step, batch in enumerate(train_loader):
        if step < skip_until:
            continue
        loss, stats = batch_loss(batch)
        loss = loss / GRAD_ACCUM
        loss.backward()
        inst = loss.item() * GRAD_ACCUM                 # instantaneous (un-scaled) batch loss
        running += inst
        ema = inst if ema is None else 0.98 * ema + 0.02 * inst   # tracks RECENT loss, not cumulative
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            global_step += 1
            if global_step % CKPT_EVERY_STEPS == 0:
                checkpoint("latest", epoch, step + 1, best_val)
        if step % LOG_EVERY == 0:
            lrs = sched.get_last_lr()
            mem_str = ""
            if torch.cuda.is_available():
                free, total = torch.cuda.mem_get_info()
                mem_str = f" gpu_free={free/1e9:.2f}GiB"
            # ema = recent loss (what to watch); avg = cumulative since s0 (damped)
            # ETA measured from the observed rate rather than a guessed s/step, so a run that
            # cannot finish is obvious within the first minute instead of hours later.
            _done = step - skip_until + 1
            _el = time.time() - t0
            eta_str = ""
            if _done >= 25 and _el > 0:
                _sps = _el / _done
                _rem = (len(train_loader) - step - 1) * _sps / 3600
                _tot = _rem + len(train_loader) * _sps * (n_epochs - epoch) / 3600
                eta_str = f" | {_sps:.2f}s/step epoch_eta={_rem:.1f}h run_eta={_tot:.1f}h"
            acc_str = ""
            if "override_rate" in stats:
                acc_str = (f" | oracle_override={stats['override_rate']:.3f} "
                          f"acc@1={stats.get('acc@1', float('nan')):.3f} "
                          f"acc@5={stats.get('acc@5', float('nan')):.3f} "
                          f"acc@10={stats.get('acc@10', float('nan')):.3f}")
            print(f"e{epoch} s{step}/{len(train_loader)} "
                  f"ema={ema:.4f} avg={running/max(step+1,1):.4f} "
                  f"lora_lr={lrs[0]:.2e} head_lr={lrs[1]:.2e} {time.time()-t0:.0f}s{mem_str}{eta_str}{acc_str}")
    # flush any trailing grad-accum remainder
    if len(train_loader) % GRAD_ACCUM != 0:
        opt.step(); sched.step(); opt.zero_grad()

    val_res = evaluate(val_ex, name=f"val (epoch {epoch})")
    print(f"epoch {epoch}  train={running/len(train_loader):.4f}  "
          f"val_loss={val_res['loss']:.4f}  val_acc1={val_res['acc1']:.4f}")

    checkpoint("latest", epoch + 1, 0, best_val)          # always: resumable at the next epoch's start
    if val_res["loss"] < best_val:
        best_val = val_res["loss"]
        checkpoint("best", epoch + 1, 0, best_val)
    if val_res["acc1"] > best_val_acc1:
        best_val_acc1 = val_res["acc1"]
        checkpoint("best_acc1", epoch + 1, 0, best_val)

## 11 · Evaluation — restricted-logit ranking
One forward pass per example; logits at the masked target position are restricted to the POI token-id
range, ranked → Acc@1/5/10 and MRR. No iterative denoising needed for ranking metrics.

In [ ]:
# ── evaluation ──────────────────────────────────────────────────────────────
val_res  = evaluate(val_ex,  "val")
test_res = evaluate(test_ex, "test")

import json
with open(f"{OUT_DIR}/stage6b_results_{DATASET}_{EMB_CONDITION}.json", "w") as f:
    json.dump(dict(model=MODEL_NAME, condition=EMB_CONDITION, scoring_mode=SCORING_MODE,
                   emb_file=os.path.basename(find(EMB_FILE)),
                   use_curvature_alignment=USE_CURVATURE_ALIGNMENT,
                   use_acc_at_t_objective=USE_ACC_AT_T_OBJECTIVE, n_masks=N_MASKS,
                   use_social_context=USE_SOCIAL_CONTEXT,
                   subsample_frac=SUBSAMPLE_FRAC, epochs=EPOCHS,
                   n_train=len(train_ex), n_val=len(val_ex), n_test=len(test_ex),
                   d1_spearman=float(_rho) if EMB_CONDITION == "hyperbolic" else None,
                   val=val_res, test=test_res), f, indent=2)
print("saved results json")

## 11b · Group-task evaluation (real `group_examples`, same trained heads)

Reuses `evaluate()` from section 11 verbatim -- section 9b's `collate` dispatches on example
shape, so passing `group_ex["val"]`/`group_ex["test"]` is the only difference. No retraining, no
new heads: this measures how well the fine-tune generalises to the group next-POI task, against
the same `poi_head`/`poi_proj`/`poi_hyper` and the same RotH-derived `W_POI` table section 7
built.

Still open (per `docs/LBSN_HANDOFF.md`): comparing this number against score-aggregation
baselines (AVG / least-misery / most-pleasure) over the same checkpoint, no training required --
`src/hyperbolic_group.py`'s `least_misery_loss`/`GeometricAttention` are committed but not
exercised here yet. Stratify on `heterogeneity`, not on `regime` (see `build_groups.py`'s
docstring for why).


In [ ]:
# ── group-task evaluation -- same evaluate(), same heads, group_ex instead of {val,test}_ex ────
if group_ex["val"] and group_ex["test"]:
    group_val_res  = evaluate(group_ex["val"],  "group-val")
    group_test_res = evaluate(group_ex["test"], "group-test")

    with open(f"{OUT_DIR}/stage5_group_results_{DATASET}_{EMB_CONDITION}.json", "w") as f:
        json.dump(dict(model=MODEL_NAME, condition=EMB_CONDITION, scoring_mode=SCORING_MODE,
                       n_masks=N_MASKS, train_on_groups=TRAIN_ON_GROUPS,
                       group_train_frac=GROUP_TRAIN_FRAC if TRAIN_ON_GROUPS else None,
                       n_group_val=len(group_ex["val"]), n_group_test=len(group_ex["test"]),
                       group_val=group_val_res, group_test=group_test_res), f, indent=2)
    print("saved group results json")
else:
    print("[group-task] group_ex['val']/['test'] empty -- nothing to evaluate.")
